In [ ]:
import gc
import os
import psutil
from PIL import Image

# Parent directory containing class folders
parent_dir = 'D:/GCN/Brain_Tumor/four_class'

def find_and_fix_images(directory):
    corrupted_count = 0
    fixed_count = 0
    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)
        if file_path.lower().endswith(('.jpeg', '.jpg', '.jpeg', '.png' '.bmp')):
            try:
                # Open the image
                with Image.open(file_path) as img:
                    # Check for alpha channel
                    if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
                        print(f"Fixing alpha channel: {file_path}")
                        img = img.convert('RGB')  # Convert to RGB
                        img.save(file_path)  # Save back without alpha channel
                        fixed_count += 1
                    # Verify image integrity
                    img.verify()
            except (IOError, SyntaxError) as e:
                print(f"Removing corrupted file: {file_path}")
                os.remove(file_path)
                corrupted_count += 1
    return corrupted_count, fixed_count

def process_all_subdirectories(parent_directory):

    total_corrupted = 0
    total_fixed = 0
    for sub_dir in os.listdir(parent_directory):
        sub_dir_path = os.path.join(parent_directory, sub_dir)
        if os.path.isdir(sub_dir_path):  # Ensure it's a directory
            print(f"Processing images in: {sub_dir_path}")
            corrupted_in_dir, fixed_in_dir = find_and_fix_images(sub_dir_path)
            print(f"Corrupted files removed from {sub_dir_path}: {corrupted_in_dir}")
            print(f"Images fixed (alpha channel removed) in {sub_dir_path}: {fixed_in_dir}")
            total_corrupted += corrupted_in_dir
            total_fixed += fixed_in_dir
    print(f"Total corrupted files removed: {total_corrupted}")
    print(f"Total images fixed (alpha channel removed): {total_fixed}")

# Start processing
process_all_subdirectories(parent_dir)



***
<a name='import Packages'>
    
# 1 <span style='color:blue'>|</span> Hybrid based KNN graph building model

In [ ]:
import gc
import os
import psutil

# Function to check memory usage
def memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024**2:.2f} MB")

# Function to clear TensorFlow cache (if using TensorFlow)
def clear_tensorflow_cache():
    try:
        import tensorflow as tf
        print("Clearing TensorFlow cache...")
        tf.keras.backend.clear_session()
    except ImportError:
        print("TensorFlow is not installed.")

# Function to clear PyTorch cache (if using PyTorch)
def clear_pytorch_cache():
    try:
        import torch
        print("Clearing PyTorch cache...")
        torch.cuda.empty_cache()
    except ImportError:
        print("PyTorch is not installed.")

# Force garbage collection
def clear_memory():
    print("Clearing memory and garbage collection...")
    gc.collect()

# Main function to clear cache and memory
def clear_cache_and_memory():
    print("Before clearing:")
    memory_usage()

    clear_tensorflow_cache()
    clear_pytorch_cache()
    clear_memory()

    print("After clearing:")
    memory_usage()

# Example usage
if __name__ == "__main__":
    clear_cache_and_memory()


In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    accuracy_score,
    log_loss
)
from sklearn.manifold import TSNE

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter
# ======================
# CONFIG
# ======================
data_dir = r"D:/GCN/Brain_Tumor/four_class"

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 123

AUGMENT = True
LR_CNN = 1e-3
EPOCHS_CNN = 30

K_DEFAULT = 12
USE_PCA = True
PCA_DIM = 256

LR_GCN = 0.005
EPOCHS_GCN = 200
DROPOUT = 0.3

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ======================
# LOAD DATA
# ======================
class_names = sorted([
    d for d in os.listdir(data_dir)
    if os.path.isdir(os.path.join(data_dir, d))
])

class_to_idx = {c: i for i, c in enumerate(class_names)}

paths, labels = [], []

for c in class_names:
    for p in glob.glob(os.path.join(data_dir, c, "*")):
        if p.lower().endswith((".jpg", ".png", ".jpeg", ".bmp", ".tif", ".tiff")):
            paths.append(p)
            labels.append(class_to_idx[c])

paths = np.array(paths)
labels = np.array(labels, dtype=np.int32)

N = len(paths)
num_classes = len(class_names)

print("Images:", N)
print("Classes:", num_classes)
print("Class names:", class_names)


# ======================
# SPLIT: 70 TRAIN, 20 VAL, 10 TEST
# ======================
idx = np.arange(N)

idx_temp, idx_te = train_test_split(
    idx,
    test_size=0.10,
    random_state=SEED,
    stratify=labels
)

idx_tr, idx_va = train_test_split(
    idx_temp,
    test_size=0.2222,
    random_state=SEED,
    stratify=labels[idx_temp]
)

mask_tr = np.zeros(N, dtype=bool)
mask_va = np.zeros(N, dtype=bool)
mask_te = np.zeros(N, dtype=bool)

mask_tr[idx_tr] = True
mask_va[idx_va] = True
mask_te[idx_te] = True

print("Train:", len(idx_tr))
print("Validation:", len(idx_va))
print("Test:", len(idx_te))


# ======================
# DATA PIPELINE
# ======================
def load_img(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img, label


def augment_img(img, label):
    if AUGMENT:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        k = tf.random.uniform([], 0, 4, dtype=tf.int32)
        img = tf.image.rot90(img, k)
    return img, label


def make_ds(idxs, training=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths[idxs], labels[idxs]))

    if shuffle:
        ds = ds.shuffle(len(idxs), seed=SEED)

    ds = ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.map(augment_img, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_ds(idx_tr, training=True, shuffle=True)
val_ds = make_ds(idx_va, training=False, shuffle=False)
all_ds = make_ds(idx, training=False, shuffle=False)


# ======================
# CNN FEATURE EXTRACTOR
# CNN is used only to learn image features
# ======================
def build_cnn():
    inp = Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    feat = layers.Dense(512, activation="relu", name="feat")(x)
    out = layers.Dropout(0.5)(feat)
    out = layers.Dense(num_classes, activation="softmax")(out)

    cnn = Model(inp, out)
    backbone = Model(inp, feat)

    return cnn, backbone


cnn, backbone = build_cnn()

cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_CNN),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_CNN,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    ],
    verbose=1
)


# ======================
# EXTRACT CNN FEATURES
# ======================
def extract_features(ds):
    X, Y = [], []

    for xb, yb in ds:
        feat = backbone(xb, training=False).numpy()
        X.append(feat)
        Y.append(yb.numpy())

    return np.vstack(X), np.concatenate(Y)


X_all, y_all = extract_features(all_ds)

print("CNN feature shape:", X_all.shape)


# ======================
# PCA + STANDARDIZATION
# Important: fit PCA and scaler only on TRAIN data
# ======================
X_tr = X_all[idx_tr]
X_va = X_all[idx_va]
X_te = X_all[idx_te]

if USE_PCA:
    pca_dim = min(PCA_DIM, X_tr.shape[1])
    pca = PCA(n_components=pca_dim, random_state=SEED)
    X_tr = pca.fit_transform(X_tr)
    X_va = pca.transform(X_va)
    X_te = pca.transform(X_te)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_va = scaler.transform(X_va)
X_te = scaler.transform(X_te)

F = X_tr.shape[1]

X_std = np.zeros((N, F), dtype=np.float32)
X_std[idx_tr] = X_tr
X_std[idx_va] = X_va
X_std[idx_te] = X_te

print("Final feature shape for graph:", X_std.shape)


# ======================
# kNN GRAPH BUILDING
# kNN builds graph from CNN features
# ======================
nbrs = NearestNeighbors(
    n_neighbors=K_DEFAULT + 1,
    metric="cosine"
).fit(X_std)

dist, idx_knn = nbrs.kneighbors(X_std)

rows, cols, data = [], [], []

for i in range(N):
    for j, d in zip(idx_knn[i], dist[i]):
        if i == j:
            continue

        sim = 1.0 - float(d)

        if sim <= 0:
            continue

        rows.append(i)
        cols.append(j)
        data.append(sim)

A = coo_matrix((data, (rows, cols)), shape=(N, N), dtype=np.float32)

# Make graph undirected
A = (A + A.T).tocsr()

# Add self loops
A.setdiag(1.0)

# Normalize for GCN
A_norm = gcn_filter(A)

print("\n===== GRAPH STATS =====")
nnz_total = A.nnz
self_loops = N
undirected_edges = (nnz_total - self_loops) // 2
degrees = np.array(A.sum(axis=1)).flatten() - 1

print("Edges:", undirected_edges)
print("Avg degree:", degrees.mean())
print("Min degree:", degrees.min())
print("Max degree:", degrees.max())

n_comp, labels_comp = connected_components(A, directed=False)
print("Components:", n_comp)


# ======================
# OPTIONAL t-SNE VISUALIZATION
# ======================
print("[t-SNE] computing...")

sample_N = min(3000, N)
sample_idx = np.random.choice(N, sample_N, replace=False)

X_ts = X_std[sample_idx]
y_ts = labels[sample_idx]

if X_ts.shape[1] > 50:
    X_ts = PCA(50, random_state=SEED).fit_transform(X_ts)

X_2d = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=SEED
).fit_transform(X_ts)

plt.figure(figsize=(7, 6))
for i, c in enumerate(class_names):
    m = y_ts == i
    plt.scatter(X_2d[m, 0], X_2d[m, 1], s=8, label=c)

plt.title("t-SNE of CNN Features")
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(figsize=(5, 4))
plt.hist(degrees, bins=40, edgecolor="black")
plt.title("Graph Degree Distribution")
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


# ======================
# GCN CLASSIFIER
# GCN performs final classification
# ======================
def build_gcn(F):
    X_in = Input(shape=(F,))
    A_in = Input(shape=(N, N), sparse=True)

    h = GCNConv(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(5e-4)
    )([X_in, A_in])

    h = layers.Dropout(DROPOUT)(h)

    out = GCNConv(
        num_classes,
        activation="softmax"
    )([h, A_in])

    model = Model([X_in, A_in], out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_GCN),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        weighted_metrics=["accuracy"]
    )
    return model
Y = to_categorical(labels, num_classes).astype(np.float32)
gcn = build_gcn(X_std.shape[1])

# ======================
# CALLBACK FOR GCN CURVES
# ======================
class GraphMetricsCallback(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()

        self.train_acc = []
        self.val_acc = []
        self.test_acc = []

        self.train_loss = []
        self.val_loss = []
        self.test_loss = []

    def on_epoch_end(self, epoch, logs=None):
        pred_prob_epoch = self.model.predict(
            [X_std, A_norm],
            batch_size=N,
            verbose=0
        )

        pred_epoch = np.argmax(pred_prob_epoch, axis=1)

        self.train_acc.append(
            accuracy_score(labels[mask_tr], pred_epoch[mask_tr])
        )
        self.val_acc.append(
            accuracy_score(labels[mask_va], pred_epoch[mask_va])
        )
        self.test_acc.append(
            accuracy_score(labels[mask_te], pred_epoch[mask_te])
        )

        self.train_loss.append(
            log_loss(
                labels[mask_tr],
                pred_prob_epoch[mask_tr],
                labels=np.arange(num_classes)
            )
        )
        self.val_loss.append(
            log_loss(
                labels[mask_va],
                pred_prob_epoch[mask_va],
                labels=np.arange(num_classes)
            )
        )
        self.test_loss.append(
            log_loss(
                labels[mask_te],
                pred_prob_epoch[mask_te],
                labels=np.arange(num_classes)
            )
        )

        print(
            f" | custom_train_acc: {self.train_acc[-1]:.4f}"
            f" | custom_val_acc: {self.val_acc[-1]:.4f}"
            f" | custom_test_acc: {self.test_acc[-1]:.4f}"
        )

metrics_cb = GraphMetricsCallback()
# ======================
# TRAIN GCN
# Only TRAIN labels are used
# Validation labels are only used for validation
# Test labels are not used for training
# ======================
history = gcn.fit(
    [X_std, A_norm],
    Y,
    sample_weight=mask_tr.astype(np.float32),
    validation_data=(
        [X_std, A_norm],
        Y,
        mask_va.astype(np.float32)
    ),
    epochs=EPOCHS_GCN,
    batch_size=N,
    shuffle=False,
    verbose=1,
    callbacks=[
        metrics_cb,
        EarlyStopping(
            monitor="val_loss",
            patience=30,
            restore_best_weights=True
        )
    ]
)


# ======================
# FINAL PREDICTION
# ======================
pred_prob = gcn.predict(
    [X_std, A_norm],
    batch_size=N,
    verbose=0
)

pred = np.argmax(pred_prob, axis=1)

# ======================
# EVALUATION FUNCTION
# ======================
def evaluate_split(split_name, mask):
    print(f"\n===== {split_name} CLASSIFICATION REPORT =====")
    print(classification_report(
        labels[mask],
        pred[mask],
        target_names=class_names
    ))

    cm = confusion_matrix(labels[mask], pred[mask])

    plt.figure(figsize=(5, 5))
    ax = sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        square=True,
        linewidths=2,
        linecolor="white",
        cbar=False,
        xticklabels=class_names,
        yticklabels=class_names,
        annot_kws={"size": 10, "color": "red"}
    )

    ax.set_title(f"{split_name} Confusion Matrix", fontsize=12, weight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

    plt.xticks(rotation=45, fontsize=8)
    plt.yticks(rotation=45, fontsize=8)
    plt.tight_layout()
    plt.show()

    y_true_bin = label_binarize(
        labels[mask],
        classes=np.arange(num_classes)
    )

    y_score = pred_prob[mask]

    plt.figure(figsize=(5, 5))

    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(
            fpr,
            tpr,
            label=f"{class_names[i]} AUC = {roc_auc:.3f}"
        )

    plt.plot([0, 1], [0, 1], "k--")
    plt.title(f"{split_name} ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


# ======================
# VALIDATION RESULTS
# ======================
evaluate_split("Validation", mask_va)


# ======================
# TEST RESULTS
# ======================
evaluate_split("Test", mask_te)


# ======================
# ACCURACY CURVE
# ======================
epochs = range(1, len(metrics_cb.train_acc) + 1)

plt.figure(figsize=(5, 4))
plt.plot(epochs, metrics_cb.train_acc, label="Train Accuracy")
plt.plot(epochs, metrics_cb.val_acc, label="Validation Accuracy")
plt.plot(epochs, metrics_cb.test_acc, label="Test Accuracy")
plt.title("GCN Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()


# ======================
# LOSS CURVE
# ======================
plt.figure(figsize=(5, 4))
plt.plot(epochs, metrics_cb.train_loss, label="Train Loss")
plt.plot(epochs, metrics_cb.val_loss, label="Validation Loss")
plt.plot(epochs, metrics_cb.test_loss, label="Test Loss")
plt.title("GCN Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FULL CORRECTED COMPUTATIONAL METRICS + DESCRIPTIVE STATS CODE
# ============================================================

import os
import gc
import time
import psutil
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy import stats


# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "CNN-GCN Framework"

# IMPORTANT:
# For your CNN-GCN code, use cnn here.
# Do NOT use model = "KNN" because that is only a string.
# The model must be a trained Keras model with .predict().
model_to_measure = cnn

input_shape = (150, 150, 3)
batch_size = 32


# ============================================================
# MEMORY HELPERS
# ============================================================

def get_memory_usage_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)


def get_model_params(model):
    return model.count_params()


def get_model_size_mb(model):
    return get_model_params(model) * 4 / (1024 * 1024)


# ============================================================
# FLOPs ESTIMATION HELPERS
# Approximate FLOPs for Conv2D, DepthwiseConv2D, Dense, BN
# ============================================================

def safe_shape(tensor):
    try:
        return tensor.shape.as_list()
    except Exception:
        try:
            return list(tensor.shape)
        except Exception:
            return None


def calculate_flops_per_image(model):
    """
    Approximate FLOPs per image.
    This is a practical estimate, not an official profiler result.
    """
    total_flops = 0

    for layer in model.layers:
        try:
            # Handle nested models, for example transfer learning backbones
            if isinstance(layer, tf.keras.Model):
                total_flops += calculate_flops_per_image(layer)
                continue

            layer_flops = 0

            if isinstance(layer, tf.keras.layers.Conv2D):
                out_shape = safe_shape(layer.output)
                kernel_h, kernel_w = layer.kernel_size
                in_channels = int(layer.kernel.shape[-2])
                out_channels = int(layer.kernel.shape[-1])

                if out_shape is not None and len(out_shape) == 4:
                    out_h = out_shape[1]
                    out_w = out_shape[2]

                    if out_h is not None and out_w is not None:
                        layer_flops = (
                            2 * out_h * out_w *
                            kernel_h * kernel_w *
                            in_channels * out_channels
                        )

            elif isinstance(layer, tf.keras.layers.DepthwiseConv2D):
                out_shape = safe_shape(layer.output)
                kernel_h, kernel_w = layer.kernel_size
                in_channels = int(layer.depthwise_kernel.shape[-2])
                depth_multiplier = int(layer.depthwise_kernel.shape[-1])

                if out_shape is not None and len(out_shape) == 4:
                    out_h = out_shape[1]
                    out_w = out_shape[2]

                    if out_h is not None and out_w is not None:
                        layer_flops = (
                            2 * out_h * out_w *
                            kernel_h * kernel_w *
                            in_channels * depth_multiplier
                        )

            elif isinstance(layer, tf.keras.layers.Dense):
                kernel_shape = layer.kernel.shape

                if kernel_shape is not None:
                    in_features = int(kernel_shape[0])
                    out_features = int(kernel_shape[1])
                    layer_flops = 2 * in_features * out_features

            elif isinstance(layer, tf.keras.layers.BatchNormalization):
                out_shape = safe_shape(layer.output)

                if out_shape is not None:
                    units = np.prod([d for d in out_shape[1:] if d is not None])
                    layer_flops = 2 * units

            total_flops += layer_flops

        except Exception:
            continue

    return int(total_flops)


# ============================================================
# INFERENCE METRICS
# ============================================================

def measure_inference_metrics(model, batch_size, input_shape, warmup_runs=3, measured_runs=10):
    """
    Measures approximate computational cost and inference speed.
    Works for CNN/Keras image models.
    """

    if not hasattr(model, "predict"):
        raise TypeError(
            "model_to_measure must be a trained Keras model, not a string. "
            "Use cnn, backbone, or another trained Keras model."
        )

    dummy_input = np.random.rand(batch_size, *input_shape).astype(np.float32)

    # Warm-up
    for _ in range(warmup_runs):
        _ = model.predict(dummy_input, verbose=0)

    gc.collect()

    start_mem = get_memory_usage_mb()
    start_time = time.time()

    for _ in range(measured_runs):
        _ = model.predict(dummy_input, verbose=0)

    end_time = time.time()
    end_mem = get_memory_usage_mb()

    total_time = end_time - start_time
    latency_ms_per_img = total_time * 1000.0 / (batch_size * measured_runs)

    flops_per_image = calculate_flops_per_image(model)
    gflops_per_image = flops_per_image / 1e9

    flops_per_batch = flops_per_image * batch_size
    gflops_per_batch = flops_per_batch / 1e9

    params = get_model_params(model)
    size_mb = get_model_size_mb(model)

    return {
        "Params": int(params),
        "Params (M)": float(params / 1e6),
        "Model Size (MB)": float(size_mb),
        "Inference Latency (ms/img)": float(latency_ms_per_img),
        "FLOPs/Image (G)": float(gflops_per_image),
        "FLOPs/Batch (G)": float(gflops_per_batch),
        "Inference Memory Delta (MB)": float(end_mem - start_mem),
    }


# ============================================================
# STATISTICAL HELPERS
# ============================================================

def descriptive_statistics(data):
    data = np.array(data, dtype=float)

    if data.size == 0:
        return (np.nan, np.nan, np.nan, np.nan, np.nan)

    mean = np.mean(data)
    std = np.std(data, ddof=1) if len(data) > 1 else 0.0
    median = np.median(data)
    iqr = np.percentile(data, 75) - np.percentile(data, 25)
    rng = np.ptp(data)

    return float(mean), float(std), float(median), float(iqr), float(rng)


def confidence_interval(data):
    data = np.array(data, dtype=float)

    if len(data) < 2:
        return (np.nan, np.nan)

    mean = np.mean(data)
    sem = stats.sem(data)

    if sem == 0 or np.isnan(sem):
        return (float(mean), float(mean))

    low, high = stats.t.interval(
        0.95,
        len(data) - 1,
        loc=mean,
        scale=sem
    )

    return float(low), float(high)


def one_sample_ttest(data, baseline=0):
    data = np.array(data, dtype=float)

    if len(data) < 2:
        return (np.nan, np.nan)

    t, p = stats.ttest_1samp(data, baseline)
    return float(t), float(p)


def effect_size(data, baseline=0):
    data = np.array(data, dtype=float)

    if len(data) < 2:
        return (np.nan, np.nan)

    mean = np.mean(data)
    std = np.std(data, ddof=1)

    if std == 0 or np.isnan(std):
        return (np.nan, np.nan)

    cohen_d = (mean - baseline) / std
    n = len(data)

    if n > 2:
        hedges_g = cohen_d * (1 - (3 / (4 * n - 9)))
    else:
        hedges_g = cohen_d

    return float(cohen_d), float(hedges_g)


def normality_test(data):
    data = np.array(data, dtype=float)

    if len(data) < 3:
        return (np.nan, np.nan)

    W, p = stats.shapiro(data)
    return float(W), float(p)


def wilcoxon_test(data, baseline=0):
    data = np.array(data, dtype=float)

    if len(data) < 3:
        return (np.nan, np.nan)

    try:
        stat, p = stats.wilcoxon(data - baseline)
        return float(stat), float(p)
    except Exception:
        return (np.nan, np.nan)


def correlation(acc, loss):
    acc = np.array(acc, dtype=float)
    loss = np.array(loss, dtype=float)

    if len(acc) < 2 or len(loss) < 2:
        return (np.nan, np.nan, np.nan, np.nan)

    min_len = min(len(acc), len(loss))
    acc = acc[:min_len]
    loss = loss[:min_len]

    try:
        pearson_r, pearson_p = stats.pearsonr(acc, loss)
    except Exception:
        pearson_r, pearson_p = np.nan, np.nan

    try:
        spearman_rho, spearman_p = stats.spearmanr(acc, loss)
    except Exception:
        spearman_rho, spearman_p = np.nan, np.nan

    return float(pearson_r), float(pearson_p), float(spearman_rho), float(spearman_p)


# ============================================================
# GET TRAINING CURVES
# ============================================================
# For your CNN-GCN code:
# metrics_cb.train_acc and metrics_cb.train_loss are preferred.
# If metrics_cb does not exist, it falls back to history.history.

if "metrics_cb" in globals():
    train_acc_final = metrics_cb.train_acc
    train_loss_final = metrics_cb.train_loss

    val_acc_final = getattr(metrics_cb, "val_acc", [])
    val_loss_final = getattr(metrics_cb, "val_loss", [])

else:
    train_acc_final = history.history.get("accuracy", [])
    train_loss_final = history.history.get("loss", [])

    val_acc_final = history.history.get("val_accuracy", [])
    val_loss_final = history.history.get("val_loss", [])


# ============================================================
# COMPUTE INFERENCE METRICS
# ============================================================

inference_metrics = measure_inference_metrics(
    model_to_measure,
    batch_size=batch_size,
    input_shape=input_shape
)


# ============================================================
# COMPUTE TRAINING DESCRIPTIVE STATS
# ============================================================

acc_mean, acc_std, acc_median, acc_iqr, acc_range = descriptive_statistics(train_acc_final)
loss_mean, loss_std, loss_median, loss_iqr, loss_range = descriptive_statistics(train_loss_final)

acc_ci = confidence_interval(train_acc_final)
loss_ci = confidence_interval(train_loss_final)

acc_t, acc_p = one_sample_ttest(train_acc_final, baseline=0)
loss_t, loss_p = one_sample_ttest(train_loss_final, baseline=0)

acc_d, acc_g = effect_size(train_acc_final, baseline=0)
loss_d, loss_g = effect_size(train_loss_final, baseline=0)

acc_W, acc_Wp = normality_test(train_acc_final)
loss_W, loss_Wp = normality_test(train_loss_final)

acc_wilc, acc_wilc_p = wilcoxon_test(train_acc_final, baseline=0)
loss_wilc, loss_wilc_p = wilcoxon_test(train_loss_final, baseline=0)

pearson_r, pearson_p, spearman_rho, spearman_p = correlation(
    train_acc_final,
    train_loss_final
)


# ============================================================
# OPTIONAL VALIDATION STATS
# ============================================================

val_acc_mean, val_acc_std, val_acc_median, val_acc_iqr, val_acc_range = descriptive_statistics(val_acc_final)
val_loss_mean, val_loss_std, val_loss_median, val_loss_iqr, val_loss_range = descriptive_statistics(val_loss_final)


# ============================================================
# COMPILE RESULTS
# ============================================================

compute_metrics = inference_metrics

compute_metrics["Train Stats"] = {
    "Accuracy Mean": acc_mean,
    "Accuracy Std": acc_std,
    "Accuracy Median": acc_median,
    "Accuracy IQR": acc_iqr,
    "Accuracy Range": acc_range,

    "Loss Mean": loss_mean,
    "Loss Std": loss_std,
    "Loss Median": loss_median,
    "Loss IQR": loss_iqr,
    "Loss Range": loss_range,

    "Accuracy CI": acc_ci,
    "Loss CI": loss_ci,

    "Accuracy T-test": (acc_t, acc_p),
    "Loss T-test": (loss_t, loss_p),

    "Accuracy Effect Size": (acc_d, acc_g),
    "Loss Effect Size": (loss_d, loss_g),

    "Accuracy Normality Test": (acc_W, acc_Wp),
    "Loss Normality Test": (loss_W, loss_Wp),

    "Accuracy Wilcoxon Test": (acc_wilc, acc_wilc_p),
    "Loss Wilcoxon Test": (loss_wilc, loss_wilc_p),

    "Accuracy-Loss Correlation": (
        pearson_r,
        pearson_p,
        spearman_rho,
        spearman_p
    ),
}

compute_metrics["Validation Stats"] = {
    "Validation Accuracy Mean": val_acc_mean,
    "Validation Accuracy Std": val_acc_std,
    "Validation Accuracy Median": val_acc_median,
    "Validation Accuracy IQR": val_acc_iqr,
    "Validation Accuracy Range": val_acc_range,

    "Validation Loss Mean": val_loss_mean,
    "Validation Loss Std": val_loss_std,
    "Validation Loss Median": val_loss_median,
    "Validation Loss IQR": val_loss_iqr,
    "Validation Loss Range": val_loss_range,
}


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n--- Model Computational Metrics and Descriptive Stats ---")
for k, v in compute_metrics.items():
    print(f"{k}: {v}")


# ============================================================
# COMPUTATIONAL COST DATAFRAME
# ============================================================

compute_metrics_df = pd.DataFrame({
    "Model": [MODEL_NAME],
    "Params": [compute_metrics["Params"]],
    "Params (M)": [compute_metrics["Params (M)"]],
    "Model Size (MB)": [compute_metrics["Model Size (MB)"]],
    "Inference Latency (ms/img)": [compute_metrics["Inference Latency (ms/img)"]],
    "FLOPs/Image (G)": [compute_metrics["FLOPs/Image (G)"]],
    "FLOPs/Batch (G)": [compute_metrics["FLOPs/Batch (G)"]],
    "Inference Memory Delta (MB)": [compute_metrics["Inference Memory Delta (MB)"]],
})

compute_metrics_df = compute_metrics_df.round({
    "Params (M)": 3,
    "Model Size (MB)": 3,
    "Inference Latency (ms/img)": 3,
    "FLOPs/Image (G)": 3,
    "FLOPs/Batch (G)": 3,
    "Inference Memory Delta (MB)": 2,
})


# ============================================================
# DESCRIPTIVE STATS DATAFRAME
# ============================================================

descriptive_stats_df = pd.DataFrame({
    "Model": [MODEL_NAME],

    "Train Acc Mean": [compute_metrics["Train Stats"]["Accuracy Mean"]],
    "Train Acc Std": [compute_metrics["Train Stats"]["Accuracy Std"]],
    "Train Acc Median": [compute_metrics["Train Stats"]["Accuracy Median"]],
    "Train Acc IQR": [compute_metrics["Train Stats"]["Accuracy IQR"]],
    "Train Acc Range": [compute_metrics["Train Stats"]["Accuracy Range"]],

    "Train Loss Mean": [compute_metrics["Train Stats"]["Loss Mean"]],
    "Train Loss Std": [compute_metrics["Train Stats"]["Loss Std"]],
    "Train Loss Median": [compute_metrics["Train Stats"]["Loss Median"]],
    "Train Loss IQR": [compute_metrics["Train Stats"]["Loss IQR"]],
    "Train Loss Range": [compute_metrics["Train Stats"]["Loss Range"]],

    "Val Acc Mean": [compute_metrics["Validation Stats"]["Validation Accuracy Mean"]],
    "Val Acc Std": [compute_metrics["Validation Stats"]["Validation Accuracy Std"]],
    "Val Acc Median": [compute_metrics["Validation Stats"]["Validation Accuracy Median"]],
    "Val Acc IQR": [compute_metrics["Validation Stats"]["Validation Accuracy IQR"]],
    "Val Acc Range": [compute_metrics["Validation Stats"]["Validation Accuracy Range"]],

    "Val Loss Mean": [compute_metrics["Validation Stats"]["Validation Loss Mean"]],
    "Val Loss Std": [compute_metrics["Validation Stats"]["Validation Loss Std"]],
    "Val Loss Median": [compute_metrics["Validation Stats"]["Validation Loss Median"]],
    "Val Loss IQR": [compute_metrics["Validation Stats"]["Validation Loss IQR"]],
    "Val Loss Range": [compute_metrics["Validation Stats"]["Validation Loss Range"]],

    "Train Acc 95% CI": [compute_metrics["Train Stats"]["Accuracy CI"]],
    "Train Loss 95% CI": [compute_metrics["Train Stats"]["Loss CI"]],

    "Train Acc T-test p-value": [compute_metrics["Train Stats"]["Accuracy T-test"][1]],
    "Train Loss T-test p-value": [compute_metrics["Train Stats"]["Loss T-test"][1]],

    "Train Acc Cohen's d": [compute_metrics["Train Stats"]["Accuracy Effect Size"][0]],
    "Train Loss Cohen's d": [compute_metrics["Train Stats"]["Loss Effect Size"][0]],

    "Train Acc Normality p-value": [compute_metrics["Train Stats"]["Accuracy Normality Test"][1]],
    "Train Loss Normality p-value": [compute_metrics["Train Stats"]["Loss Normality Test"][1]],

    "Train Acc Wilcoxon p-value": [compute_metrics["Train Stats"]["Accuracy Wilcoxon Test"][1]],
    "Train Loss Wilcoxon p-value": [compute_metrics["Train Stats"]["Loss Wilcoxon Test"][1]],

    "Acc-Loss Pearson r": [compute_metrics["Train Stats"]["Accuracy-Loss Correlation"][0]],
    "Acc-Loss Pearson p-value": [compute_metrics["Train Stats"]["Accuracy-Loss Correlation"][1]],
    "Acc-Loss Spearman rho": [compute_metrics["Train Stats"]["Accuracy-Loss Correlation"][2]],
    "Acc-Loss Spearman p-value": [compute_metrics["Train Stats"]["Accuracy-Loss Correlation"][3]],
})

descriptive_stats_df = descriptive_stats_df.round(4)


# ============================================================
# FINAL COMBINED DATAFRAME
# ============================================================

final_df = pd.concat(
    [
        compute_metrics_df,
        descriptive_stats_df.drop(columns=["Model"])
    ],
    axis=1
)

display(final_df)


# ============================================================
# OPTIONAL: SAVE RESULTS
# ============================================================

final_df.to_csv("computational_cost_and_descriptive_statistics.csv", index=False)

print("\nSaved as: computational_cost_and_descriptive_statistics.csv")


***
<a name='import Packages'>
    
# 2 <span style='color:blue'>|</span> Hybrid ε-graph building model

In [ ]:
import gc
import os
import psutil

# Function to check memory usage
def memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024**2:.2f} MB")

# Function to clear TensorFlow cache (if using TensorFlow)
def clear_tensorflow_cache():
    try:
        import tensorflow as tf
        print("Clearing TensorFlow cache...")
        tf.keras.backend.clear_session()
    except ImportError:
        print("TensorFlow is not installed.")

# Function to clear PyTorch cache (if using PyTorch)
def clear_pytorch_cache():
    try:
        import torch
        print("Clearing PyTorch cache...")
        torch.cuda.empty_cache()
    except ImportError:
        print("PyTorch is not installed.")

# Force garbage collection
def clear_memory():
    print("Clearing memory and garbage collection...")
    gc.collect()

# Main function to clear cache and memory
def clear_cache_and_memory():
    print("Before clearing:")
    memory_usage()

    clear_tensorflow_cache()
    clear_pytorch_cache()
    clear_memory()

    print("After clearing:")
    memory_usage()

# Example usage
if __name__ == "__main__":
    clear_cache_and_memory()


In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    accuracy_score,
    log_loss
)
from sklearn.manifold import TSNE

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter
# ======================
# CONFIG
# ======================
data_dir = r"D:/GCN/Brain_Tumor/four_class"

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 123

AUGMENT = True
LR_CNN = 1e-3
EPOCHS_CNN = 30

K_DEFAULT = 12
USE_PCA = True
PCA_DIM = 256

LR_GCN = 0.005
EPOCHS_GCN = 200
DROPOUT = 0.3

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ======================
# LOAD DATA
# ======================
class_names = sorted([
    d for d in os.listdir(data_dir)
    if os.path.isdir(os.path.join(data_dir, d))
])

class_to_idx = {c: i for i, c in enumerate(class_names)}

paths, labels = [], []

for c in class_names:
    for p in glob.glob(os.path.join(data_dir, c, "*")):
        if p.lower().endswith((".jpg", ".png", ".jpeg", ".bmp", ".tif", ".tiff")):
            paths.append(p)
            labels.append(class_to_idx[c])

paths = np.array(paths)
labels = np.array(labels, dtype=np.int32)

N = len(paths)
num_classes = len(class_names)

print("Images:", N)
print("Classes:", num_classes)
print("Class names:", class_names)


# ======================
# SPLIT: 70 TRAIN, 20 VAL, 10 TEST
# ======================
idx = np.arange(N)

idx_temp, idx_te = train_test_split(
    idx,
    test_size=0.10,
    random_state=SEED,
    stratify=labels
)

idx_tr, idx_va = train_test_split(
    idx_temp,
    test_size=0.2222,
    random_state=SEED,
    stratify=labels[idx_temp]
)

mask_tr = np.zeros(N, dtype=bool)
mask_va = np.zeros(N, dtype=bool)
mask_te = np.zeros(N, dtype=bool)

mask_tr[idx_tr] = True
mask_va[idx_va] = True
mask_te[idx_te] = True

print("Train:", len(idx_tr))
print("Validation:", len(idx_va))
print("Test:", len(idx_te))


# ======================
# DATA PIPELINE
# ======================
def load_img(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img, label


def augment_img(img, label):
    if AUGMENT:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        k = tf.random.uniform([], 0, 4, dtype=tf.int32)
        img = tf.image.rot90(img, k)
    return img, label


def make_ds(idxs, training=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths[idxs], labels[idxs]))

    if shuffle:
        ds = ds.shuffle(len(idxs), seed=SEED)

    ds = ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.map(augment_img, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_ds(idx_tr, training=True, shuffle=True)
val_ds = make_ds(idx_va, training=False, shuffle=False)
all_ds = make_ds(idx, training=False, shuffle=False)


# ======================
# CNN FEATURE EXTRACTOR
# CNN is used only to learn image features
# ======================
def build_cnn():
    inp = Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    feat = layers.Dense(512, activation="relu", name="feat")(x)
    out = layers.Dropout(0.5)(feat)
    out = layers.Dense(num_classes, activation="softmax")(out)

    cnn = Model(inp, out)
    backbone = Model(inp, feat)

    return cnn, backbone


cnn, backbone = build_cnn()

cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_CNN),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_CNN,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    ],
    verbose=1
)


# ======================
# EXTRACT CNN FEATURES
# ======================
def extract_features(ds):
    X, Y = [], []

    for xb, yb in ds:
        feat = backbone(xb, training=False).numpy()
        X.append(feat)
        Y.append(yb.numpy())

    return np.vstack(X), np.concatenate(Y)


X_all, y_all = extract_features(all_ds)

print("CNN feature shape:", X_all.shape)


# ======================
# PCA + STANDARDIZATION
# Important: fit PCA and scaler only on TRAIN data
# ======================
X_tr = X_all[idx_tr]
X_va = X_all[idx_va]
X_te = X_all[idx_te]

if USE_PCA:
    pca_dim = min(PCA_DIM, X_tr.shape[1])
    pca = PCA(n_components=pca_dim, random_state=SEED)
    X_tr = pca.fit_transform(X_tr)
    X_va = pca.transform(X_va)
    X_te = pca.transform(X_te)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_va = scaler.transform(X_va)
X_te = scaler.transform(X_te)

F = X_tr.shape[1]

X_std = np.zeros((N, F), dtype=np.float32)
X_std[idx_tr] = X_tr
X_std[idx_va] = X_va
X_std[idx_te] = X_te

print("Final feature shape for graph:", X_std.shape)

# ======================
# EPSILON GRAPH BUILDING
# Epsilon graph builds graph from CNN features
# ======================

EPSILON = 0.85   # cosine similarity threshold
                 # higher = fewer edges
                 # lower  = more edges

def build_graph_epsilon_cosine(X, eps=EPSILON):
    radius = max(1e-6, 1.0 - float(eps))

    N_local = X.shape[0]

    nn = NearestNeighbors(
        metric="cosine",
        radius=radius
    ).fit(X)

    nbrs = nn.radius_neighbors(
        X,
        return_distance=False
    )

    rows, cols, data = [], [], []

    for i, neighbors in enumerate(nbrs):
        for j in neighbors:
            if i == j:
                continue

            rows.append(i)
            cols.append(j)
            data.append(1.0)

    A_dir = coo_matrix(
        (data, (rows, cols)),
        shape=(N_local, N_local),
        dtype=np.float32
    )

    # Mutual epsilon graph
    A = A_dir.minimum(A_dir.T).tocsr()

    # Add self loops
    A.setdiag(1.0)

    return A


A = build_graph_epsilon_cosine(X_std, eps=EPSILON)

# Normalize for GCN
A_norm = gcn_filter(A)


print("\n===== GRAPH STATS =====")
nnz_total = A.nnz
self_loops = N
undirected_edges = (nnz_total - self_loops) // 2
degrees = np.array(A.sum(axis=1)).flatten() - 1

print("Edges:", undirected_edges)
print("Avg degree:", degrees.mean())
print("Min degree:", degrees.min())
print("Max degree:", degrees.max())

n_comp, labels_comp = connected_components(A, directed=False)
print("Components:", n_comp)


# ======================
# OPTIONAL t-SNE VISUALIZATION
# ======================
print("[t-SNE] computing...")

sample_N = min(3000, N)
sample_idx = np.random.choice(N, sample_N, replace=False)

X_ts = X_std[sample_idx]
y_ts = labels[sample_idx]

if X_ts.shape[1] > 50:
    X_ts = PCA(50, random_state=SEED).fit_transform(X_ts)

X_2d = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=SEED
).fit_transform(X_ts)

plt.figure(figsize=(7, 6))
for i, c in enumerate(class_names):
    m = y_ts == i
    plt.scatter(X_2d[m, 0], X_2d[m, 1], s=8, label=c)

plt.title("t-SNE of CNN Features")
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(figsize=(5, 4))
plt.hist(degrees, bins=40, edgecolor="black")
plt.title("Graph Degree Distribution")
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


# ======================
# GCN CLASSIFIER
# GCN performs final classification
# ======================
def build_gcn(F):
    X_in = Input(shape=(F,))
    A_in = Input(shape=(N, N), sparse=True)

    h = GCNConv(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(5e-4)
    )([X_in, A_in])

    h = layers.Dropout(DROPOUT)(h)

    out = GCNConv(
        num_classes,
        activation="softmax"
    )([h, A_in])

    model = Model([X_in, A_in], out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_GCN),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        weighted_metrics=["accuracy"]
    )
    return model
Y = to_categorical(labels, num_classes).astype(np.float32)
gcn = build_gcn(X_std.shape[1])

# ======================
# CALLBACK FOR GCN CURVES
# ======================
class GraphMetricsCallback(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()

        self.train_acc = []
        self.val_acc = []
        self.test_acc = []

        self.train_loss = []
        self.val_loss = []
        self.test_loss = []

    def on_epoch_end(self, epoch, logs=None):
        pred_prob_epoch = self.model.predict(
            [X_std, A_norm],
            batch_size=N,
            verbose=0
        )

        pred_epoch = np.argmax(pred_prob_epoch, axis=1)

        self.train_acc.append(
            accuracy_score(labels[mask_tr], pred_epoch[mask_tr])
        )
        self.val_acc.append(
            accuracy_score(labels[mask_va], pred_epoch[mask_va])
        )
        self.test_acc.append(
            accuracy_score(labels[mask_te], pred_epoch[mask_te])
        )

        self.train_loss.append(
            log_loss(
                labels[mask_tr],
                pred_prob_epoch[mask_tr],
                labels=np.arange(num_classes)
            )
        )
        self.val_loss.append(
            log_loss(
                labels[mask_va],
                pred_prob_epoch[mask_va],
                labels=np.arange(num_classes)
            )
        )
        self.test_loss.append(
            log_loss(
                labels[mask_te],
                pred_prob_epoch[mask_te],
                labels=np.arange(num_classes)
            )
        )

        print(
            f" | custom_train_acc: {self.train_acc[-1]:.4f}"
            f" | custom_val_acc: {self.val_acc[-1]:.4f}"
            f" | custom_test_acc: {self.test_acc[-1]:.4f}"
        )

metrics_cb = GraphMetricsCallback()
# ======================
# TRAIN GCN
# Only TRAIN labels are used
# Validation labels are only used for validation
# Test labels are not used for training
# ======================
history = gcn.fit(
    [X_std, A_norm],
    Y,
    sample_weight=mask_tr.astype(np.float32),
    validation_data=(
        [X_std, A_norm],
        Y,
        mask_va.astype(np.float32)
    ),
    epochs=EPOCHS_GCN,
    batch_size=N,
    shuffle=False,
    verbose=1,
    callbacks=[
        metrics_cb,
        EarlyStopping(
            monitor="val_loss",
            patience=30,
            restore_best_weights=True
        )
    ]
)


# ======================
# FINAL PREDICTION
# ======================
pred_prob = gcn.predict(
    [X_std, A_norm],
    batch_size=N,
    verbose=0
)

pred = np.argmax(pred_prob, axis=1)

# ======================
# EVALUATION FUNCTION
# ======================
def evaluate_split(split_name, mask):
    print(f"\n===== {split_name} CLASSIFICATION REPORT =====")
    print(classification_report(
        labels[mask],
        pred[mask],
        target_names=class_names
    ))

    cm = confusion_matrix(labels[mask], pred[mask])

    plt.figure(figsize=(5, 5))
    ax = sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        square=True,
        linewidths=2,
        linecolor="white",
        cbar=False,
        xticklabels=class_names,
        yticklabels=class_names,
        annot_kws={"size": 10, "color": "red"}
    )

    ax.set_title(f"{split_name} Confusion Matrix", fontsize=12, weight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

    plt.xticks(rotation=45, fontsize=8)
    plt.yticks(rotation=45, fontsize=8)
    plt.tight_layout()
    plt.show()

    y_true_bin = label_binarize(
        labels[mask],
        classes=np.arange(num_classes)
    )

    y_score = pred_prob[mask]

    plt.figure(figsize=(5, 5))

    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(
            fpr,
            tpr,
            label=f"{class_names[i]} AUC = {roc_auc:.3f}"
        )

    plt.plot([0, 1], [0, 1], "k--")
    plt.title(f"{split_name} ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


# ======================
# VALIDATION RESULTS
# ======================
evaluate_split("Validation", mask_va)


# ======================
# TEST RESULTS
# ======================
evaluate_split("Test", mask_te)


# ======================
# ACCURACY CURVE
# ======================
epochs = range(1, len(metrics_cb.train_acc) + 1)

plt.figure(figsize=(5, 4))
plt.plot(epochs, metrics_cb.train_acc, label="Train Accuracy")
plt.plot(epochs, metrics_cb.val_acc, label="Validation Accuracy")
plt.plot(epochs, metrics_cb.test_acc, label="Test Accuracy")
plt.title("GCN Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()


# ======================
# LOSS CURVE
# ======================
plt.figure(figsize=(5, 4))
plt.plot(epochs, metrics_cb.train_loss, label="Train Loss")
plt.plot(epochs, metrics_cb.val_loss, label="Validation Loss")
plt.plot(epochs, metrics_cb.test_loss, label="Test Loss")
plt.title("GCN Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import gc
import psutil
import numpy as np
from scipy import stats
import tensorflow as tf

# =========================
# HELPERS
# =========================

def get_memory_usage_mb():
    process = psutil.Process()
    return process.memory_info().rss / (1024 * 1024)

def get_model_params(model):
    return model.count_params()

def get_model_size_mb(model):
    return get_model_params(model) * 4 / (1024 * 1024)

def safe_shape(tensor):
    try:
        return tensor.shape.as_list()
    except:
        try:
            return list(tensor.shape)
        except:
            return None

def calculate_flops_per_image(model):
    total_flops = 0
    for layer in model.layers:
        try:
            if isinstance(layer, tf.keras.Model):
                total_flops += calculate_flops_per_image(layer)
                continue
            layer_flops = 0
            if isinstance(layer, tf.keras.layers.Conv2D):
                out_shape = safe_shape(layer.output)
                kh, kw = layer.kernel_size
                in_ch = int(layer.kernel.shape[-2])
                out_ch = int(layer.kernel.shape[-1])
                if out_shape and len(out_shape)==4:
                    oh, ow = out_shape[1], out_shape[2]
                    if oh and ow:
                        layer_flops = 2*oh*ow*kh*kw*in_ch*out_ch
            elif isinstance(layer, tf.keras.layers.Dense):
                w = layer.kernel.shape
                if w: layer_flops = 2*int(w[0])*int(w[1])
            elif isinstance(layer, tf.keras.layers.BatchNormalization):
                out_shape = safe_shape(layer.output)
                if out_shape:
                    units = np.prod([d for d in out_shape[1:] if d])
                    layer_flops = 2*units
            total_flops += layer_flops
        except:
            continue
    return int(total_flops)

def measure_inference_metrics(model, batch_size, warmup_runs=3, measured_runs=10):
    # Warmup
    for _ in range(warmup_runs):
        _ = model.predict([X_std, A_norm], batch_size=N, verbose=0)
    gc.collect()
    start_mem = get_memory_usage_mb()
    start_time = time.time()
    for _ in range(measured_runs):
        _ = model.predict([X_std, A_norm], batch_size=N, verbose=0)
    end_time = time.time()
    end_mem = get_memory_usage_mb()
    latency_ms = (end_time - start_time)*1000/(N*measured_runs)
    flops_img = calculate_flops_per_image(model)
    flops_batch = flops_img * N
    params = get_model_params(model)
    size_mb = get_model_size_mb(model)
    return {
        "Params": int(params),
        "Params (M)": params/1e6,
        "Model Size (MB)": size_mb,
        "Inference Latency (ms/img)": latency_ms,
        "FLOPs/Image (G)": flops_img/1e9,
        "FLOPs/Batch (G)": flops_batch/1e9,
        "Inference Memory Delta (MB)": end_mem-start_mem
    }

# =========================
# STATISTICAL HELPERS
# =========================

def descriptive_statistics(data):
    data = np.array(data, dtype=float)
    if len(data)==0: return (np.nan,)*5
    return (
        float(np.mean(data)),
        float(np.std(data, ddof=1) if len(data)>1 else 0),
        float(np.median(data)),
        float(np.percentile(data,75)-np.percentile(data,25)),
        float(np.ptp(data))
    )

def confidence_interval(data):
    data = np.array(data, dtype=float)
    if len(data)<2: return (np.nan,np.nan)
    mean = np.mean(data)
    sem = stats.sem(data)
    low, high = stats.t.interval(0.95,len(data)-1,loc=mean,scale=sem)
    return float(low), float(high)

def one_sample_ttest(data, baseline):
    if len(data)<2: return (np.nan,np.nan)
    t,p = stats.ttest_1samp(data, baseline)
    return float(t), float(p)

def effect_size(data, baseline):
    if len(data)<2: return (np.nan,np.nan)
    mean = np.mean(data)
    std = np.std(data, ddof=1)
    if std==0: return (np.nan,np.nan)
    d = (mean-baseline)/std
    n = len(data)
    g = d*(1-(3/(4*n-9))) if n>2 else d
    return float(d), float(g)

def normality_test(data):
    if len(data)<3: return (np.nan,np.nan)
    W,p = stats.shapiro(data)
    return float(W), float(p)

def wilcoxon_test(data, baseline):
    if len(data)<3: return (np.nan,np.nan)
    stat,p = stats.wilcoxon(np.array(data)-baseline)
    return float(stat), float(p)

def correlation(acc,loss):
    if len(acc)<2 or len(loss)<2: return (np.nan,)*4
    min_len = min(len(acc), len(loss))
    acc = np.array(acc[:min_len])
    loss = np.array(loss[:min_len])
    pr,pp = stats.pearsonr(acc,loss)
    sr,sp = stats.spearmanr(acc,loss)
    return float(pr), float(pp), float(sr), float(sp)

# =========================
# COMPUTE & PRINT VERTICALLY
# =========================

baseline_acc = 1/num_classes
baseline_loss = np.log(num_classes)

inference_metrics = measure_inference_metrics(gcn, batch_size)

acc_mean,acc_std,acc_med,acc_iqr,acc_range = descriptive_statistics(metrics_cb.train_acc)
loss_mean,loss_std,loss_med,loss_iqr,loss_range = descriptive_statistics(metrics_cb.train_loss)
acc_ci = confidence_interval(metrics_cb.train_acc)
loss_ci = confidence_interval(metrics_cb.train_loss)
acc_t,acc_p = one_sample_ttest(metrics_cb.train_acc, baseline_acc)
loss_t,loss_p = one_sample_ttest(metrics_cb.train_loss, baseline_loss)
acc_d,acc_g = effect_size(metrics_cb.train_acc, baseline_acc)
loss_d,loss_g = effect_size(metrics_cb.train_loss, baseline_loss)
acc_W,acc_Wp = normality_test(metrics_cb.train_acc)
loss_W,loss_Wp = normality_test(metrics_cb.train_loss)
acc_w,acc_wp = wilcoxon_test(metrics_cb.train_acc, baseline_acc)
loss_w,loss_wp = wilcoxon_test(metrics_cb.train_loss, baseline_loss)
pearson_r,pearson_p,spearman_rho,spearman_p = correlation(metrics_cb.train_acc, metrics_cb.train_loss)

print("\n=== COMPUTATIONAL METRICS ===")
for k,v in inference_metrics.items():
    print(f"{k}: {v}")

print("\n=== TRAIN STATISTICS ===")
print(f"Train Acc Mean: {acc_mean:.4f}")
print(f"Train Acc Std: {acc_std:.4f}")
print(f"Train Acc 95% CI: {acc_ci}")
print(f"Train Acc t-test p-value: {acc_p:.4f}")
print(f"Train Acc Cohen d: {acc_d:.4f}")
print(f"Train Acc Normality p-value: {acc_Wp:.4f}")
print(f"Train Acc Wilcoxon p-value: {acc_wp:.4f}")

print(f"Train Loss Mean: {loss_mean:.4f}")
print(f"Train Loss Std: {loss_std:.4f}")
print(f"Train Loss 95% CI: {loss_ci}")
print(f"Train Loss t-test p-value: {loss_p:.4f}")
print(f"Train Loss Cohen d: {loss_d:.4f}")
print(f"Train Loss Normality p-value: {loss_Wp:.4f}")
print(f"Train Loss Wilcoxon p-value: {loss_wp:.4f}")

print(f"Accuracy-Loss Pearson r: {pearson_r:.4f}")
print(f"Accuracy-Loss Pearson p-value: {pearson_p:.4f}")
print(f"Accuracy-Loss Spearman rho: {spearman_rho:.4f}")
print(f"Accuracy-Loss Spearman p-value: {spearman_p:.4f}")




# ============================================================
# INFERENCE METRICS
# ============================================================

def measure_inference_metrics(model, batch_size, input_shape, warmup_runs=3, measured_runs=10):
    """
    Measures approximate computational cost and inference speed.
    Works for CNN/Keras image models.
    """

    if not hasattr(model, "predict"):
        raise TypeError(
            "model_to_measure must be a trained Keras model, not a string. "
            "Use cnn, backbone, or another trained Keras model."
        )

    dummy_input = np.random.rand(batch_size, *input_shape).astype(np.float32)

    # Warm-up
    for _ in range(warmup_runs):
        _ = model.predict(dummy_input, verbose=0)

    gc.collect()

    start_mem = get_memory_usage_mb()
    start_time = time.time()

    for _ in range(measured_runs):
        _ = model.predict(dummy_input, verbose=0)

    end_time = time.time()
    end_mem = get_memory_usage_mb()

    total_time = end_time - start_time
    latency_ms_per_img = total_time * 1000.0 / (batch_size * measured_runs)

    flops_per_image = calculate_flops_per_image(model)
    gflops_per_image = flops_per_image / 1e9

    flops_per_batch = flops_per_image * batch_size
    gflops_per_batch = flops_per_batch / 1e9

    params = get_model_params(model)
    size_mb = get_model_size_mb(model)

    return {
        "Params": int(params),
        "Params (M)": float(params / 1e6),
        "Model Size (MB)": float(size_mb),
        "Inference Latency (ms/img)": float(latency_ms_per_img),
        "FLOPs/Image (G)": float(gflops_per_image),
        "FLOPs/Batch (G)": float(gflops_per_batch),
        "Inference Memory Delta (MB)": float(end_mem - start_mem),
    }


# ============================================================
# STATISTICAL HELPERS
# ============================================================

def descriptive_statistics(data):
    data = np.array(data, dtype=float)

    if data.size == 0:
        return (np.nan, np.nan, np.nan, np.nan, np.nan)

    mean = np.mean(data)
    std = np.std(data, ddof=1) if len(data) > 1 else 0.0
    median = np.median(data)
    iqr = np.percentile(data, 75) - np.percentile(data, 25)
    rng = np.ptp(data)

    return float(mean), float(std), float(median), float(iqr), float(rng)


def confidence_interval(data):
    data = np.array(data, dtype=float)

    if len(data) < 2:
        return (np.nan, np.nan)

    mean = np.mean(data)
    sem = stats.sem(data)

    if sem == 0 or np.isnan(sem):
        return (float(mean), float(mean))

    low, high = stats.t.interval(
        0.95,
        len(data) - 1,
        loc=mean,
        scale=sem
    )

    return float(low), float(high)


def one_sample_ttest(data, baseline=0):
    data = np.array(data, dtype=float)

    if len(data) < 2:
        return (np.nan, np.nan)

    t, p = stats.ttest_1samp(data, baseline)
    return float(t), float(p)


def effect_size(data, baseline=0):
    data = np.array(data, dtype=float)

    if len(data) < 2:
        return (np.nan, np.nan)

    mean = np.mean(data)
    std = np.std(data, ddof=1)

    if std == 0 or np.isnan(std):
        return (np.nan, np.nan)

    cohen_d = (mean - baseline) / std
    n = len(data)

    if n > 2:
        hedges_g = cohen_d * (1 - (3 / (4 * n - 9)))
    else:
        hedges_g = cohen_d

    return float(cohen_d), float(hedges_g)


def normality_test(data):
    data = np.array(data, dtype=float)

    if len(data) < 3:
        return (np.nan, np.nan)

    W, p = stats.shapiro(data)
    return float(W), float(p)


def wilcoxon_test(data, baseline=0):
    data = np.array(data, dtype=float)

    if len(data) < 3:
        return (np.nan, np.nan)

    try:
        stat, p = stats.wilcoxon(data - baseline)
        return float(stat), float(p)
    except Exception:
        return (np.nan, np.nan)


def correlation(acc, loss):
    acc = np.array(acc, dtype=float)
    loss = np.array(loss, dtype=float)

    if len(acc) < 2 or len(loss) < 2:
        return (np.nan, np.nan, np.nan, np.nan)

    min_len = min(len(acc), len(loss))
    acc = acc[:min_len]
    loss = loss[:min_len]

    try:
        pearson_r, pearson_p = stats.pearsonr(acc, loss)
    except Exception:
        pearson_r, pearson_p = np.nan, np.nan

    try:
        spearman_rho, spearman_p = stats.spearmanr(acc, loss)
    except Exception:
        spearman_rho, spearman_p = np.nan, np.nan

    return float(pearson_r), float(pearson_p), float(spearman_rho), float(spearman_p)


# ============================================================
# GET TRAINING CURVES
# ============================================================
# For your CNN-GCN code:
# metrics_cb.train_acc and metrics_cb.train_loss are preferred.
# If metrics_cb does not exist, it falls back to history.history.

if "metrics_cb" in globals():
    train_acc_final = metrics_cb.train_acc
    train_loss_final = metrics_cb.train_loss

    val_acc_final = getattr(metrics_cb, "val_acc", [])
    val_loss_final = getattr(metrics_cb, "val_loss", [])

else:
    train_acc_final = history.history.get("accuracy", [])
    train_loss_final = history.history.get("loss", [])

    val_acc_final = history.history.get("val_accuracy", [])
    val_loss_final = history.history.get("val_loss", [])


# ============================================================
# COMPUTE INFERENCE METRICS
# ============================================================

inference_metrics = measure_inference_metrics(
    model_to_measure,
    batch_size=batch_size,
    input_shape=input_shape
)


# ============================================================
# COMPUTE TRAINING DESCRIPTIVE STATS
# ============================================================

acc_mean, acc_std, acc_median, acc_iqr, acc_range = descriptive_statistics(train_acc_final)
loss_mean, loss_std, loss_median, loss_iqr, loss_range = descriptive_statistics(train_loss_final)

acc_ci = confidence_interval(train_acc_final)
loss_ci = confidence_interval(train_loss_final)

acc_t, acc_p = one_sample_ttest(train_acc_final, baseline=0)
loss_t, loss_p = one_sample_ttest(train_loss_final, baseline=0)

acc_d, acc_g = effect_size(train_acc_final, baseline=0)
loss_d, loss_g = effect_size(train_loss_final, baseline=0)

acc_W, acc_Wp = normality_test(train_acc_final)
loss_W, loss_Wp = normality_test(train_loss_final)

acc_wilc, acc_wilc_p = wilcoxon_test(train_acc_final, baseline=0)
loss_wilc, loss_wilc_p = wilcoxon_test(train_loss_final, baseline=0)

pearson_r, pearson_p, spearman_rho, spearman_p = correlation(
    train_acc_final,
    train_loss_final
)


# ============================================================
# OPTIONAL VALIDATION STATS
# ============================================================

val_acc_mean, val_acc_std, val_acc_median, val_acc_iqr, val_acc_range = descriptive_statistics(val_acc_final)
val_loss_mean, val_loss_std, val_loss_median, val_loss_iqr, val_loss_range = descriptive_statistics(val_loss_final)


# ============================================================
# COMPILE RESULTS
# ============================================================

compute_metrics = inference_metrics

compute_metrics["Train Stats"] = {
    "Accuracy Mean": acc_mean,
    "Accuracy Std": acc_std,
    "Accuracy Median": acc_median,
    "Accuracy IQR": acc_iqr,
    "Accuracy Range": acc_range,

    "Loss Mean": loss_mean,
    "Loss Std": loss_std,
    "Loss Median": loss_median,
    "Loss IQR": loss_iqr,
    "Loss Range": loss_range,

    "Accuracy CI": acc_ci,
    "Loss CI": loss_ci,

    "Accuracy T-test": (acc_t, acc_p),
    "Loss T-test": (loss_t, loss_p),

    "Accuracy Effect Size": (acc_d, acc_g),
    "Loss Effect Size": (loss_d, loss_g),

    "Accuracy Normality Test": (acc_W, acc_Wp),
    "Loss Normality Test": (loss_W, loss_Wp),

    "Accuracy Wilcoxon Test": (acc_wilc, acc_wilc_p),
    "Loss Wilcoxon Test": (loss_wilc, loss_wilc_p),

    "Accuracy-Loss Correlation": (
        pearson_r,
        pearson_p,
        spearman_rho,
        spearman_p
    ),
}

compute_metrics["Validation Stats"] = {
    "Validation Accuracy Mean": val_acc_mean,
    "Validation Accuracy Std": val_acc_std,
    "Validation Accuracy Median": val_acc_median,
    "Validation Accuracy IQR": val_acc_iqr,
    "Validation Accuracy Range": val_acc_range,

    "Validation Loss Mean": val_loss_mean,
    "Validation Loss Std": val_loss_std,
    "Validation Loss Median": val_loss_median,
    "Validation Loss IQR": val_loss_iqr,
    "Validation Loss Range": val_loss_range,
}


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n--- Model Computational Metrics and Descriptive Stats ---")
for k, v in compute_metrics.items():
    print(f"{k}: {v}")


# ============================================================
# COMPUTATIONAL COST DATAFRAME
# ============================================================

compute_metrics_df = pd.DataFrame({
    "Model": [MODEL_NAME],
    "Params": [compute_metrics["Params"]],
    "Params (M)": [compute_metrics["Params (M)"]],
    "Model Size (MB)": [compute_metrics["Model Size (MB)"]],
    "Inference Latency (ms/img)": [compute_metrics["Inference Latency (ms/img)"]],
    "FLOPs/Image (G)": [compute_metrics["FLOPs/Image (G)"]],
    "FLOPs/Batch (G)": [compute_metrics["FLOPs/Batch (G)"]],
    "Inference Memory Delta (MB)": [compute_metrics["Inference Memory Delta (MB)"]],
})

compute_metrics_df = compute_metrics_df.round({
    "Params (M)": 3,
    "Model Size (MB)": 3,
    "Inference Latency (ms/img)": 3,
    "FLOPs/Image (G)": 3,
    "FLOPs/Batch (G)": 3,
    "Inference Memory Delta (MB)": 2,
})


# ============================================================
# DESCRIPTIVE STATS DATAFRAME
# ============================================================

descriptive_stats_df = pd.DataFrame({
    "Model": [MODEL_NAME],

    "Train Acc Mean": [compute_metrics["Train Stats"]["Accuracy Mean"]],
    "Train Acc Std": [compute_metrics["Train Stats"]["Accuracy Std"]],
    "Train Acc Median": [compute_metrics["Train Stats"]["Accuracy Median"]],
    "Train Acc IQR": [compute_metrics["Train Stats"]["Accuracy IQR"]],
    "Train Acc Range": [compute_metrics["Train Stats"]["Accuracy Range"]],

    "Train Loss Mean": [compute_metrics["Train Stats"]["Loss Mean"]],
    "Train Loss Std": [compute_metrics["Train Stats"]["Loss Std"]],
    "Train Loss Median": [compute_metrics["Train Stats"]["Loss Median"]],
    "Train Loss IQR": [compute_metrics["Train Stats"]["Loss IQR"]],
    "Train Loss Range": [compute_metrics["Train Stats"]["Loss Range"]],

    "Val Acc Mean": [compute_metrics["Validation Stats"]["Validation Accuracy Mean"]],
    "Val Acc Std": [compute_metrics["Validation Stats"]["Validation Accuracy Std"]],
    "Val Acc Median": [compute_metrics["Validation Stats"]["Validation Accuracy Median"]],
    "Val Acc IQR": [compute_metrics["Validation Stats"]["Validation Accuracy IQR"]],
    "Val Acc Range": [compute_metrics["Validation Stats"]["Validation Accuracy Range"]],

    "Val Loss Mean": [compute_metrics["Validation Stats"]["Validation Loss Mean"]],
    "Val Loss Std": [compute_metrics["Validation Stats"]["Validation Loss Std"]],
    "Val Loss Median": [compute_metrics["Validation Stats"]["Validation Loss Median"]],
    "Val Loss IQR": [compute_metrics["Validation Stats"]["Validation Loss IQR"]],
    "Val Loss Range": [compute_metrics["Validation Stats"]["Validation Loss Range"]],

    "Train Acc 95% CI": [compute_metrics["Train Stats"]["Accuracy CI"]],
    "Train Loss 95% CI": [compute_metrics["Train Stats"]["Loss CI"]],

    "Train Acc T-test p-value": [compute_metrics["Train Stats"]["Accuracy T-test"][1]],
    "Train Loss T-test p-value": [compute_metrics["Train Stats"]["Loss T-test"][1]],

    "Train Acc Cohen's d": [compute_metrics["Train Stats"]["Accuracy Effect Size"][0]],
    "Train Loss Cohen's d": [compute_metrics["Train Stats"]["Loss Effect Size"][0]],

    "Train Acc Normality p-value": [compute_metrics["Train Stats"]["Accuracy Normality Test"][1]],
    "Train Loss Normality p-value": [compute_metrics["Train Stats"]["Loss Normality Test"][1]],

    "Train Acc Wilcoxon p-value": [compute_metrics["Train Stats"]["Accuracy Wilcoxon Test"][1]],
    "Train Loss Wilcoxon p-value": [compute_metrics["Train Stats"]["Loss Wilcoxon Test"][1]],

    "Acc-Loss Pearson r": [compute_metrics["Train Stats"]["Accuracy-Loss Correlation"][0]],
    "Acc-Loss Pearson p-value": [compute_metrics["Train Stats"]["Accuracy-Loss Correlation"][1]],
    "Acc-Loss Spearman rho": [compute_metrics["Train Stats"]["Accuracy-Loss Correlation"][2]],
    "Acc-Loss Spearman p-value": [compute_metrics["Train Stats"]["Accuracy-Loss Correlation"][3]],
})

descriptive_stats_df = descriptive_stats_df.round(4)


# ============================================================
# FINAL COMBINED DATAFRAME
# ============================================================

final_df = pd.concat(
    [
        compute_metrics_df,
        descriptive_stats_df.drop(columns=["Model"])
    ],
    axis=1
)

display(final_df)


# ============================================================
# OPTIONAL: SAVE RESULTS
# ============================================================

final_df.to_csv("computational_cost_and_descriptive_statistics.csv", index=False)

print("\nSaved as: computational_cost_and_descriptive_statistics.csv")

In [ ]:
import os
import gc
import time
import psutil
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy import stats

# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "CNN-GCN Framework"

# Choose which model to measure
# Options: "CNN" or "GCN"
MEASURE_PART = "GCN"

if MEASURE_PART == "CNN":
    model_to_measure = cnn
    input_shape = (150, 150, 3)

elif MEASURE_PART == "GCN":
    model_to_measure = gcn
    input_shape = None  # not used for graph model

else:
    raise ValueError("MEASURE_PART must be 'CNN' or 'GCN'")

batch_size = 32


# ============================================================
# MEMORY + PARAM HELPERS
# ============================================================

def get_memory_usage_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)


def get_model_params(model):
    return model.count_params()


def get_model_size_mb(model):
    return get_model_params(model) * 4 / (1024 * 1024)


# ============================================================
# FLOPs ESTIMATION (Approximate)
# ============================================================

def safe_shape(tensor):
    try:
        return tensor.shape.as_list()
    except Exception:
        try:
            return list(tensor.shape)
        except Exception:
            return None


def calculate_flops_per_image(model):
    total_flops = 0

    for layer in model.layers:
        try:
            if isinstance(layer, tf.keras.Model):
                total_flops += calculate_flops_per_image(layer)
                continue

            layer_flops = 0

            if isinstance(layer, tf.keras.layers.Conv2D):
                out_shape = safe_shape(layer.output)
                kernel_h, kernel_w = layer.kernel_size
                in_channels = int(layer.kernel.shape[-2])
                out_channels = int(layer.kernel.shape[-1])

                if out_shape and len(out_shape) == 4:
                    out_h = out_shape[1]
                    out_w = out_shape[2]
                    if out_h and out_w:
                        layer_flops = (
                            2 * out_h * out_w *
                            kernel_h * kernel_w *
                            in_channels * out_channels
                        )

            elif isinstance(layer, tf.keras.layers.DepthwiseConv2D):
                out_shape = safe_shape(layer.output)
                kernel_h, kernel_w = layer.kernel_size
                in_channels = int(layer.depthwise_kernel.shape[-2])
                depth_multiplier = int(layer.depthwise_kernel.shape[-1])

                if out_shape and len(out_shape) == 4:
                    out_h = out_shape[1]
                    out_w = out_shape[2]
                    if out_h and out_w:
                        layer_flops = (
                            2 * out_h * out_w *
                            kernel_h * kernel_w *
                            in_channels * depth_multiplier
                        )

            elif isinstance(layer, tf.keras.layers.Dense):
                kernel_shape = layer.kernel.shape
                if kernel_shape:
                    in_features = int(kernel_shape[0])
                    out_features = int(kernel_shape[1])
                    layer_flops = 2 * in_features * out_features

            elif isinstance(layer, tf.keras.layers.BatchNormalization):
                out_shape = safe_shape(layer.output)
                if out_shape:
                    units = np.prod([d for d in out_shape[1:] if d])
                    layer_flops = 2 * units

            total_flops += layer_flops

        except Exception:
            continue

    return int(total_flops)


# ============================================================
# INFERENCE METRICS
# ============================================================

def measure_inference_metrics(model, batch_size,
                              warmup_runs=3, measured_runs=10):

    if not hasattr(model, "predict"):
        raise TypeError("Model must be trained Keras model.")

    # -------------------------
    # CNN case
    # -------------------------
    if MEASURE_PART == "CNN":

        dummy_input = np.random.rand(
            batch_size, *input_shape
        ).astype(np.float32)

        for _ in range(warmup_runs):
            _ = model.predict(dummy_input, verbose=0)

        gc.collect()

        start_mem = get_memory_usage_mb()
        start_time = time.time()

        for _ in range(measured_runs):
            _ = model.predict(dummy_input, verbose=0)

        end_time = time.time()
        end_mem = get_memory_usage_mb()

        latency_ms = (
            (end_time - start_time) * 1000
            / (batch_size * measured_runs)
        )

        flops_per_image = calculate_flops_per_image(model)
        flops_per_batch = flops_per_image * batch_size

    # -------------------------
    # GCN case (full graph)
    # -------------------------
    elif MEASURE_PART == "GCN":

        for _ in range(warmup_runs):
            _ = model.predict([X_std, A_norm],
                              batch_size=N,
                              verbose=0)

        gc.collect()

        start_mem = get_memory_usage_mb()
        start_time = time.time()

        for _ in range(measured_runs):
            _ = model.predict([X_std, A_norm],
                              batch_size=N,
                              verbose=0)

        end_time = time.time()
        end_mem = get_memory_usage_mb()

        total_time = end_time - start_time
        latency_ms = (total_time * 1000) / (N * measured_runs)

        flops_per_image = calculate_flops_per_image(model)
        flops_per_batch = flops_per_image * N

    else:
        raise ValueError("Invalid MEASURE_PART")

    params = get_model_params(model)
    size_mb = get_model_size_mb(model)

    return {
        "Params": int(params),
        "Params (M)": params / 1e6,
        "Model Size (MB)": size_mb,
        "Inference Latency (ms/img)": latency_ms,
        "FLOPs/Image (G)": flops_per_image / 1e9,
        "FLOPs/Batch (G)": flops_per_batch / 1e9,
        "Inference Memory Delta (MB)": end_mem - start_mem,
    }


# ============================================================
# DESCRIPTIVE STATISTICS FUNCTIONS
# ============================================================

def descriptive_statistics(data):
    data = np.array(data, dtype=float)
    if len(data) == 0:
        return (np.nan,)*5

    return (
        float(np.mean(data)),
        float(np.std(data, ddof=1) if len(data)>1 else 0),
        float(np.median(data)),
        float(np.percentile(data,75)-np.percentile(data,25)),
        float(np.ptp(data))
    )


def confidence_interval(data):
    data = np.array(data, dtype=float)
    if len(data)<2:
        return (np.nan,np.nan)
    mean=np.mean(data)
    sem=stats.sem(data)
    low,high=stats.t.interval(0.95,len(data)-1,loc=mean,scale=sem)
    return float(low),float(high)


def one_sample_ttest(data,baseline):
    if len(data)<2:
        return (np.nan,np.nan)
    t,p=stats.ttest_1samp(data,baseline)
    return float(t),float(p)


def effect_size(data,baseline):
    if len(data)<2:
        return (np.nan,np.nan)
    mean=np.mean(data)
    std=np.std(data,ddof=1)
    if std==0: return (np.nan,np.nan)
    d=(mean-baseline)/std
    n=len(data)
    g=d*(1-(3/(4*n-9))) if n>2 else d
    return float(d),float(g)


def normality_test(data):
    if len(data)<3:
        return (np.nan,np.nan)
    W,p=stats.shapiro(data)
    return float(W),float(p)


def wilcoxon_test(data,baseline):
    if len(data)<3:
        return (np.nan,np.nan)
    stat,p=stats.wilcoxon(np.array(data)-baseline)
    return float(stat),float(p)


def correlation(acc,loss):
    if len(acc)<2 or len(loss)<2:
        return (np.nan,)*4
    min_len=min(len(acc),len(loss))
    acc=np.array(acc[:min_len])
    loss=np.array(loss[:min_len])
    pr,pp=stats.pearsonr(acc,loss)
    sr,sp=stats.spearmanr(acc,loss)
    return float(pr),float(pp),float(sr),float(sp)


# ============================================================
# GET TRAINING CURVES
# ============================================================

if "metrics_cb" in globals():
    train_acc_final = metrics_cb.train_acc
    train_loss_final = metrics_cb.train_loss
    val_acc_final = metrics_cb.val_acc
    val_loss_final = metrics_cb.val_loss
else:
    train_acc_final = history.history.get("accuracy",[])
    train_loss_final = history.history.get("loss",[])
    val_acc_final = history.history.get("val_accuracy",[])
    val_loss_final = history.history.get("val_loss",[])


# Correct statistical baselines
baseline_acc = 1 / num_classes
baseline_loss = np.log(num_classes)


# ============================================================
# COMPUTE METRICS
# ============================================================

inference_metrics = measure_inference_metrics(model_to_measure,batch_size)

acc_mean,acc_std,acc_median,acc_iqr,acc_range=descriptive_statistics(train_acc_final)
loss_mean,loss_std,loss_median,loss_iqr,loss_range=descriptive_statistics(train_loss_final)

acc_ci=confidence_interval(train_acc_final)
loss_ci=confidence_interval(train_loss_final)

acc_t,acc_p=one_sample_ttest(train_acc_final,baseline_acc)
loss_t,loss_p=one_sample_ttest(train_loss_final,baseline_loss)

acc_d,acc_g=effect_size(train_acc_final,baseline_acc)
loss_d,loss_g=effect_size(train_loss_final,baseline_loss)

acc_W,acc_Wp=normality_test(train_acc_final)
loss_W,loss_Wp=normality_test(train_loss_final)

acc_w,acc_wp=wilcoxon_test(train_acc_final,baseline_acc)
loss_w,loss_wp=wilcoxon_test(train_loss_final,baseline_loss)

pearson_r,pearson_p,spearman_rho,spearman_p=correlation(train_acc_final,train_loss_final)


# ============================================================
# FINAL DATAFRAME
# ============================================================

compute_df=pd.DataFrame([inference_metrics]).round(4)

stats_df=pd.DataFrame({
    "Train Acc Mean":[acc_mean],
    "Train Acc Std":[acc_std],
    "Train Acc 95% CI":[acc_ci],
    "Train Acc p-value":[acc_p],
    "Train Acc Cohen d":[acc_d],
    "Train Loss Mean":[loss_mean],
    "Train Loss Std":[loss_std],
    "Train Loss 95% CI":[loss_ci],
    "Train Loss p-value":[loss_p],
    "Acc-Loss Pearson r":[pearson_r],
    "Acc-Loss Spearman rho":[spearman_rho]
}).round(4)




# ============================================================
# INFERENCE METRICS
# ============================================================

def measure_inference_metrics(model, batch_size, input_shape, warmup_runs=3, measured_runs=10):
    """
    Measures approximate computational cost and inference speed.
    Works for CNN/Keras image models.
    """

    if not hasattr(model, "predict"):
        raise TypeError(
            "model_to_measure must be a trained Keras model, not a string. "
            "Use cnn, backbone, or another trained Keras model."
        )

    dummy_input = np.random.rand(batch_size, *input_shape).astype(np.float32)

    # Warm-up
    for _ in range(warmup_runs):
        _ = model.predict(dummy_input, verbose=0)

    gc.collect()

    start_mem = get_memory_usage_mb()
    start_time = time.time()

    for _ in range(measured_runs):
        _ = model.predict(dummy_input, verbose=0)

    end_time = time.time()
    end_mem = get_memory_usage_mb()

    total_time = end_time - start_time
    latency_ms_per_img = total_time * 1000.0 / (batch_size * measured_runs)

    flops_per_image = calculate_flops_per_image(model)
    gflops_per_image = flops_per_image / 1e9

    flops_per_batch = flops_per_image * batch_size
    gflops_per_batch = flops_per_batch / 1e9

    params = get_model_params(model)
    size_mb = get_model_size_mb(model)

    return {
        "Params": int(params),
        "Params (M)": float(params / 1e6),
        "Model Size (MB)": float(size_mb),
        "Inference Latency (ms/img)": float(latency_ms_per_img),
        "FLOPs/Image (G)": float(gflops_per_image),
        "FLOPs/Batch (G)": float(gflops_per_batch),
        "Inference Memory Delta (MB)": float(end_mem - start_mem),
    }


# ============================================================
# STATISTICAL HELPERS
# ============================================================

def descriptive_statistics(data):
    data = np.array(data, dtype=float)

    if data.size == 0:
        return (np.nan, np.nan, np.nan, np.nan, np.nan)

    mean = np.mean(data)
    std = np.std(data, ddof=1) if len(data) > 1 else 0.0
    median = np.median(data)
    iqr = np.percentile(data, 75) - np.percentile(data, 25)
    rng = np.ptp(data)

    return float(mean), float(std), float(median), float(iqr), float(rng)


def confidence_interval(data):
    data = np.array(data, dtype=float)

    if len(data) < 2:
        return (np.nan, np.nan)

    mean = np.mean(data)
    sem = stats.sem(data)

    if sem == 0 or np.isnan(sem):
        return (float(mean), float(mean))

    low, high = stats.t.interval(
        0.95,
        len(data) - 1,
        loc=mean,
        scale=sem
    )

    return float(low), float(high)


def one_sample_ttest(data, baseline=0):
    data = np.array(data, dtype=float)

    if len(data) < 2:
        return (np.nan, np.nan)

    t, p = stats.ttest_1samp(data, baseline)
    return float(t), float(p)


def effect_size(data, baseline=0):
    data = np.array(data, dtype=float)

    if len(data) < 2:
        return (np.nan, np.nan)

    mean = np.mean(data)
    std = np.std(data, ddof=1)

    if std == 0 or np.isnan(std):
        return (np.nan, np.nan)

    cohen_d = (mean - baseline) / std
    n = len(data)

    if n > 2:
        hedges_g = cohen_d * (1 - (3 / (4 * n - 9)))
    else:
        hedges_g = cohen_d

    return float(cohen_d), float(hedges_g)


def normality_test(data):
    data = np.array(data, dtype=float)

    if len(data) < 3:
        return (np.nan, np.nan)

    W, p = stats.shapiro(data)
    return float(W), float(p)


def wilcoxon_test(data, baseline=0):
    data = np.array(data, dtype=float)

    if len(data) < 3:
        return (np.nan, np.nan)

    try:
        stat, p = stats.wilcoxon(data - baseline)
        return float(stat), float(p)
    except Exception:
        return (np.nan, np.nan)


def correlation(acc, loss):
    acc = np.array(acc, dtype=float)
    loss = np.array(loss, dtype=float)

    if len(acc) < 2 or len(loss) < 2:
        return (np.nan, np.nan, np.nan, np.nan)

    min_len = min(len(acc), len(loss))
    acc = acc[:min_len]
    loss = loss[:min_len]

    try:
        pearson_r, pearson_p = stats.pearsonr(acc, loss)
    except Exception:
        pearson_r, pearson_p = np.nan, np.nan

    try:
        spearman_rho, spearman_p = stats.spearmanr(acc, loss)
    except Exception:
        spearman_rho, spearman_p = np.nan, np.nan

    return float(pearson_r), float(pearson_p), float(spearman_rho), float(spearman_p)


# ============================================================
# GET TRAINING CURVES
# ============================================================
# For your CNN-GCN code:
# metrics_cb.train_acc and metrics_cb.train_loss are preferred.
# If metrics_cb does not exist, it falls back to history.history.

if "metrics_cb" in globals():
    train_acc_final = metrics_cb.train_acc
    train_loss_final = metrics_cb.train_loss

    val_acc_final = getattr(metrics_cb, "val_acc", [])
    val_loss_final = getattr(metrics_cb, "val_loss", [])

else:
    train_acc_final = history.history.get("accuracy", [])
    train_loss_final = history.history.get("loss", [])

    val_acc_final = history.history.get("val_accuracy", [])
    val_loss_final = history.history.get("val_loss", [])


# ============================================================
# COMPUTE INFERENCE METRICS
# ============================================================




# ============================================================
# COMPUTE TRAINING DESCRIPTIVE STATS
# ============================================================

acc_mean, acc_std, acc_median, acc_iqr, acc_range = descriptive_statistics(train_acc_final)
loss_mean, loss_std, loss_median, loss_iqr, loss_range = descriptive_statistics(train_loss_final)

acc_ci = confidence_interval(train_acc_final)
loss_ci = confidence_interval(train_loss_final)

acc_t, acc_p = one_sample_ttest(train_acc_final, baseline=0)
loss_t, loss_p = one_sample_ttest(train_loss_final, baseline=0)

acc_d, acc_g = effect_size(train_acc_final, baseline=0)
loss_d, loss_g = effect_size(train_loss_final, baseline=0)

acc_W, acc_Wp = normality_test(train_acc_final)
loss_W, loss_Wp = normality_test(train_loss_final)

acc_wilc, acc_wilc_p = wilcoxon_test(train_acc_final, baseline=0)
loss_wilc, loss_wilc_p = wilcoxon_test(train_loss_final, baseline=0)

pearson_r, pearson_p, spearman_rho, spearman_p = correlation(
    train_acc_final,
    train_loss_final
)


# ============================================================
# OPTIONAL VALIDATION STATS
# ============================================================

val_acc_mean, val_acc_std, val_acc_median, val_acc_iqr, val_acc_range = descriptive_statistics(val_acc_final)
val_loss_mean, val_loss_std, val_loss_median, val_loss_iqr, val_loss_range = descriptive_statistics(val_loss_final)


# ============================================================
# COMPILE RESULTS
# ============================================================

compute_metrics = inference_metrics

compute_metrics["Train Stats"] = {
    "Accuracy Mean": acc_mean,
    "Accuracy Std": acc_std,
    "Accuracy Median": acc_median,
    "Accuracy IQR": acc_iqr,
    "Accuracy Range": acc_range,

    "Loss Mean": loss_mean,
    "Loss Std": loss_std,
    "Loss Median": loss_median,
    "Loss IQR": loss_iqr,
    "Loss Range": loss_range,

    "Accuracy CI": acc_ci,
    "Loss CI": loss_ci,

    "Accuracy T-test": (acc_t, acc_p),
    "Loss T-test": (loss_t, loss_p),

    "Accuracy Effect Size": (acc_d, acc_g),
    "Loss Effect Size": (loss_d, loss_g),

    "Accuracy Normality Test": (acc_W, acc_Wp),
    "Loss Normality Test": (loss_W, loss_Wp),

    "Accuracy Wilcoxon Test": (acc_wilc, acc_wilc_p),
    "Loss Wilcoxon Test": (loss_wilc, loss_wilc_p),

    "Accuracy-Loss Correlation": (
        pearson_r,
        pearson_p,
        spearman_rho,
        spearman_p
    ),
}

compute_metrics["Validation Stats"] = {
    "Validation Accuracy Mean": val_acc_mean,
    "Validation Accuracy Std": val_acc_std,
    "Validation Accuracy Median": val_acc_median,
    "Validation Accuracy IQR": val_acc_iqr,
    "Validation Accuracy Range": val_acc_range,

    "Validation Loss Mean": val_loss_mean,
    "Validation Loss Std": val_loss_std,
    "Validation Loss Median": val_loss_median,
    "Validation Loss IQR": val_loss_iqr,
    "Validation Loss Range": val_loss_range,
}


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n--- Model Computational Metrics and Descriptive Stats ---")
for k, v in compute_metrics.items():
    print(f"{k}: {v}")


# ============================================================
# COMPUTATIONAL COST DATAFRAME
# ============================================================

compute_metrics_df = pd.DataFrame({
    "Model": [MODEL_NAME],
    "Params": [compute_metrics["Params"]],
    "Params (M)": [compute_metrics["Params (M)"]],
    "Model Size (MB)": [compute_metrics["Model Size (MB)"]],
    "Inference Latency (ms/img)": [compute_metrics["Inference Latency (ms/img)"]],
    "FLOPs/Image (G)": [compute_metrics["FLOPs/Image (G)"]],
    "FLOPs/Batch (G)": [compute_metrics["FLOPs/Batch (G)"]],
    "Inference Memory Delta (MB)": [compute_metrics["Inference Memory Delta (MB)"]],
})

compute_metrics_df = compute_metrics_df.round({
    "Params (M)": 3,
    "Model Size (MB)": 3,
    "Inference Latency (ms/img)": 3,
    "FLOPs/Image (G)": 3,
    "FLOPs/Batch (G)": 3,
    "Inference Memory Delta (MB)": 2,
})


# ============================================================
# DESCRIPTIVE STATS DATAFRAME
# ============================================================

descriptive_stats_df = pd.DataFrame({
    "Model": [MODEL_NAME],

    "Train Acc Mean": [compute_metrics["Train Stats"]["Accuracy Mean"]],
    "Train Acc Std": [compute_metrics["Train Stats"]["Accuracy Std"]],
    "Train Acc Median": [compute_metrics["Train Stats"]["Accuracy Median"]],
    "Train Acc IQR": [compute_metrics["Train Stats"]["Accuracy IQR"]],
    "Train Acc Range": [compute_metrics["Train Stats"]["Accuracy Range"]],

    "Train Loss Mean": [compute_metrics["Train Stats"]["Loss Mean"]],
    "Train Loss Std": [compute_metrics["Train Stats"]["Loss Std"]],
    "Train Loss Median": [compute_metrics["Train Stats"]["Loss Median"]],
    "Train Loss IQR": [compute_metrics["Train Stats"]["Loss IQR"]],
    "Train Loss Range": [compute_metrics["Train Stats"]["Loss Range"]],

    "Val Acc Mean": [compute_metrics["Validation Stats"]["Validation Accuracy Mean"]],
    "Val Acc Std": [compute_metrics["Validation Stats"]["Validation Accuracy Std"]],
    "Val Acc Median": [compute_metrics["Validation Stats"]["Validation Accuracy Median"]],
    "Val Acc IQR": [compute_metrics["Validation Stats"]["Validation Accuracy IQR"]],
    "Val Acc Range": [compute_metrics["Validation Stats"]["Validation Accuracy Range"]],

    "Val Loss Mean": [compute_metrics["Validation Stats"]["Validation Loss Mean"]],
    "Val Loss Std": [compute_metrics["Validation Stats"]["Validation Loss Std"]],
    "Val Loss Median": [compute_metrics["Validation Stats"]["Validation Loss Median"]],
    "Val Loss IQR": [compute_metrics["Validation Stats"]["Validation Loss IQR"]],
    "Val Loss Range": [compute_metrics["Validation Stats"]["Validation Loss Range"]],

    "Train Acc 95% CI": [compute_metrics["Train Stats"]["Accuracy CI"]],
    "Train Loss 95% CI": [compute_metrics["Train Stats"]["Loss CI"]],

    "Train Acc T-test p-value": [compute_metrics["Train Stats"]["Accuracy T-test"][1]],
    "Train Loss T-test p-value": [compute_metrics["Train Stats"]["Loss T-test"][1]],

    "Train Acc Cohen's d": [compute_metrics["Train Stats"]["Accuracy Effect Size"][0]],
    "Train Loss Cohen's d": [compute_metrics["Train Stats"]["Loss Effect Size"][0]],

    "Train Acc Normality p-value": [compute_metrics["Train Stats"]["Accuracy Normality Test"][1]],
    "Train Loss Normality p-value": [compute_metrics["Train Stats"]["Loss Normality Test"][1]],

    "Train Acc Wilcoxon p-value": [compute_metrics["Train Stats"]["Accuracy Wilcoxon Test"][1]],
    "Train Loss Wilcoxon p-value": [compute_metrics["Train Stats"]["Loss Wilcoxon Test"][1]],

    "Acc-Loss Pearson r": [compute_metrics["Train Stats"]["Accuracy-Loss Correlation"][0]],
    "Acc-Loss Pearson p-value": [compute_metrics["Train Stats"]["Accuracy-Loss Correlation"][1]],
    "Acc-Loss Spearman rho": [compute_metrics["Train Stats"]["Accuracy-Loss Correlation"][2]],
    "Acc-Loss Spearman p-value": [compute_metrics["Train Stats"]["Accuracy-Loss Correlation"][3]],
})

descriptive_stats_df = descriptive_stats_df.round(4)


# ============================================================
# FINAL COMBINED DATAFRAME
# ============================================================

final_df = pd.concat(
    [
        compute_metrics_df,
        descriptive_stats_df.drop(columns=["Model"])
    ],
    axis=1
)

display(final_df)


# ============================================================
# OPTIONAL: SAVE RESULTS
# ============================================================

final_df.to_csv("computational_cost_and_descriptive_statistics.csv", index=False)

print("\nSaved as: computational_cost_and_descriptive_statistics.csv")
final_df=pd.concat([compute_df,stats_df],axis=1)

print("\n--- FINAL COMPUTATIONAL + STATISTICAL SUMMARY ---")
display(final_df)

final_df.to_csv("computational_cost_and_descriptive_statistics.csv",index=False)
print("\nSaved as: computational_cost_and_descriptive_statistics.csv")


***
<a name='import Packages'>
    
# 3 <span style='color:blue'>|</span> RBF 

In [ ]:
import gc
import os
import psutil

# Function to check memory usage
def memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024**2:.2f} MB")

# Function to clear TensorFlow cache (if using TensorFlow)
def clear_tensorflow_cache():
    try:
        import tensorflow as tf
        print("Clearing TensorFlow cache...")
        tf.keras.backend.clear_session()
    except ImportError:
        print("TensorFlow is not installed.")

# Function to clear PyTorch cache (if using PyTorch)
def clear_pytorch_cache():
    try:
        import torch
        print("Clearing PyTorch cache...")
        torch.cuda.empty_cache()
    except ImportError:
        print("PyTorch is not installed.")

# Force garbage collection
def clear_memory():
    print("Clearing memory and garbage collection...")
    gc.collect()

# Main function to clear cache and memory
def clear_cache_and_memory():
    print("Before clearing:")
    memory_usage()

    clear_tensorflow_cache()
    clear_pytorch_cache()
    clear_memory()

    print("After clearing:")
    memory_usage()

# Example usage
if __name__ == "__main__":
    clear_cache_and_memory()


In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    accuracy_score,
    log_loss
)
from sklearn.manifold import TSNE

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter
# ======================
# CONFIG
# ======================
data_dir = r"D:/GCN/Brain_Tumor/four_class"

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 123

AUGMENT = True
LR_CNN = 1e-3
EPOCHS_CNN = 30

K_DEFAULT = 12
USE_PCA = True
PCA_DIM = 256

LR_GCN = 0.005
EPOCHS_GCN = 200
DROPOUT = 0.3

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ======================
# LOAD DATA
# ======================
class_names = sorted([
    d for d in os.listdir(data_dir)
    if os.path.isdir(os.path.join(data_dir, d))
])

class_to_idx = {c: i for i, c in enumerate(class_names)}

paths, labels = [], []

for c in class_names:
    for p in glob.glob(os.path.join(data_dir, c, "*")):
        if p.lower().endswith((".jpg", ".png", ".jpeg", ".bmp", ".tif", ".tiff")):
            paths.append(p)
            labels.append(class_to_idx[c])

paths = np.array(paths)
labels = np.array(labels, dtype=np.int32)

N = len(paths)
num_classes = len(class_names)

print("Images:", N)
print("Classes:", num_classes)
print("Class names:", class_names)


# ======================
# SPLIT: 70 TRAIN, 20 VAL, 10 TEST
# ======================
idx = np.arange(N)

idx_temp, idx_te = train_test_split(
    idx,
    test_size=0.10,
    random_state=SEED,
    stratify=labels
)

idx_tr, idx_va = train_test_split(
    idx_temp,
    test_size=0.2222,
    random_state=SEED,
    stratify=labels[idx_temp]
)

mask_tr = np.zeros(N, dtype=bool)
mask_va = np.zeros(N, dtype=bool)
mask_te = np.zeros(N, dtype=bool)

mask_tr[idx_tr] = True
mask_va[idx_va] = True
mask_te[idx_te] = True

print("Train:", len(idx_tr))
print("Validation:", len(idx_va))
print("Test:", len(idx_te))


# ======================
# DATA PIPELINE
# ======================
def load_img(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img, label


def augment_img(img, label):
    if AUGMENT:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        k = tf.random.uniform([], 0, 4, dtype=tf.int32)
        img = tf.image.rot90(img, k)
    return img, label


def make_ds(idxs, training=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths[idxs], labels[idxs]))

    if shuffle:
        ds = ds.shuffle(len(idxs), seed=SEED)

    ds = ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.map(augment_img, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_ds(idx_tr, training=True, shuffle=True)
val_ds = make_ds(idx_va, training=False, shuffle=False)
all_ds = make_ds(idx, training=False, shuffle=False)


# ======================
# CNN FEATURE EXTRACTOR
# CNN is used only to learn image features
# ======================
def build_cnn():
    inp = Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    feat = layers.Dense(512, activation="relu", name="feat")(x)
    out = layers.Dropout(0.5)(feat)
    out = layers.Dense(num_classes, activation="softmax")(out)

    cnn = Model(inp, out)
    backbone = Model(inp, feat)

    return cnn, backbone


cnn, backbone = build_cnn()

cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_CNN),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_CNN,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    ],
    verbose=1
)


# ======================
# EXTRACT CNN FEATURES
# ======================
def extract_features(ds):
    X, Y = [], []

    for xb, yb in ds:
        feat = backbone(xb, training=False).numpy()
        X.append(feat)
        Y.append(yb.numpy())

    return np.vstack(X), np.concatenate(Y)


X_all, y_all = extract_features(all_ds)

print("CNN feature shape:", X_all.shape)


# ======================
# PCA + STANDARDIZATION
# Important: fit PCA and scaler only on TRAIN data
# ======================
X_tr = X_all[idx_tr]
X_va = X_all[idx_va]
X_te = X_all[idx_te]

if USE_PCA:
    pca_dim = min(PCA_DIM, X_tr.shape[1])
    pca = PCA(n_components=pca_dim, random_state=SEED)
    X_tr = pca.fit_transform(X_tr)
    X_va = pca.transform(X_va)
    X_te = pca.transform(X_te)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_va = scaler.transform(X_va)
X_te = scaler.transform(X_te)

F = X_tr.shape[1]

X_std = np.zeros((N, F), dtype=np.float32)
X_std[idx_tr] = X_tr
X_std[idx_va] = X_va
X_std[idx_te] = X_te

print("Final feature shape for graph:", X_std.shape)

# ======================
# RBF GRAPH BUILDING
# RBF graph builds graph from CNN features
# ======================

from scipy.sparse import csr_matrix

RBF_GAMMA = 1.0 / max(1, X_std.shape[1])
RBF_K = 12

def build_graph_rbf_kernel(X, gamma=RBF_GAMMA, k=RBF_K):
    N_local = X.shape[0]

    nn = NearestNeighbors(
        n_neighbors=k + 1,
        metric="euclidean"
    ).fit(X)

    dist, knn_idx = nn.kneighbors(
        X,
        return_distance=True
    )

    rows, cols, data = [], [], []

    for i in range(N_local):
        for j, d in zip(knn_idx[i], dist[i]):
            if i == j:
                continue

            w = np.exp(-gamma * float(d) * float(d))

            if w <= 0:
                continue

            rows.append(i)
            cols.append(j)
            data.append(w)

    A_dir = coo_matrix(
        (data, (rows, cols)),
        shape=(N_local, N_local),
        dtype=np.float32
    )

    # Make graph undirected
    A = A_dir.maximum(A_dir.T).tocsr()

    # Add self loops
    A.setdiag(1.0)

    return A


A = build_graph_rbf_kernel(
    X_std,
    gamma=RBF_GAMMA,
    k=RBF_K
)

# Normalize for GCN
A_norm = gcn_filter(A)


print("\n===== GRAPH STATS =====")
nnz_total = A.nnz
self_loops = N
undirected_edges = (nnz_total - self_loops) // 2
degrees = np.array(A.sum(axis=1)).flatten() - 1

print("Edges:", undirected_edges)
print("Avg degree:", degrees.mean())
print("Min degree:", degrees.min())
print("Max degree:", degrees.max())

n_comp, labels_comp = connected_components(A, directed=False)
print("Components:", n_comp)


# ======================
# OPTIONAL t-SNE VISUALIZATION
# ======================
print("[t-SNE] computing...")

sample_N = min(3000, N)
sample_idx = np.random.choice(N, sample_N, replace=False)

X_ts = X_std[sample_idx]
y_ts = labels[sample_idx]

if X_ts.shape[1] > 50:
    X_ts = PCA(50, random_state=SEED).fit_transform(X_ts)

X_2d = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=SEED
).fit_transform(X_ts)

plt.figure(figsize=(7, 6))
for i, c in enumerate(class_names):
    m = y_ts == i
    plt.scatter(X_2d[m, 0], X_2d[m, 1], s=8, label=c)

plt.title("t-SNE of CNN Features")
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(figsize=(5, 4))
plt.hist(degrees, bins=40, edgecolor="black")
plt.title("Graph Degree Distribution")
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


# ======================
# GCN CLASSIFIER
# GCN performs final classification
# ======================
def build_gcn(F):
    X_in = Input(shape=(F,))
    A_in = Input(shape=(N, N), sparse=True)

    h = GCNConv(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(5e-4)
    )([X_in, A_in])

    h = layers.Dropout(DROPOUT)(h)

    out = GCNConv(
        num_classes,
        activation="softmax"
    )([h, A_in])

    model = Model([X_in, A_in], out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_GCN),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        weighted_metrics=["accuracy"]
    )
    return model
Y = to_categorical(labels, num_classes).astype(np.float32)
gcn = build_gcn(X_std.shape[1])

# ======================
# CALLBACK FOR GCN CURVES
# ======================
class GraphMetricsCallback(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()

        self.train_acc = []
        self.val_acc = []
        self.test_acc = []

        self.train_loss = []
        self.val_loss = []
        self.test_loss = []

    def on_epoch_end(self, epoch, logs=None):
        pred_prob_epoch = self.model.predict(
            [X_std, A_norm],
            batch_size=N,
            verbose=0
        )

        pred_epoch = np.argmax(pred_prob_epoch, axis=1)

        self.train_acc.append(
            accuracy_score(labels[mask_tr], pred_epoch[mask_tr])
        )
        self.val_acc.append(
            accuracy_score(labels[mask_va], pred_epoch[mask_va])
        )
        self.test_acc.append(
            accuracy_score(labels[mask_te], pred_epoch[mask_te])
        )

        self.train_loss.append(
            log_loss(
                labels[mask_tr],
                pred_prob_epoch[mask_tr],
                labels=np.arange(num_classes)
            )
        )
        self.val_loss.append(
            log_loss(
                labels[mask_va],
                pred_prob_epoch[mask_va],
                labels=np.arange(num_classes)
            )
        )
        self.test_loss.append(
            log_loss(
                labels[mask_te],
                pred_prob_epoch[mask_te],
                labels=np.arange(num_classes)
            )
        )

        print(
            f" | custom_train_acc: {self.train_acc[-1]:.4f}"
            f" | custom_val_acc: {self.val_acc[-1]:.4f}"
            f" | custom_test_acc: {self.test_acc[-1]:.4f}"
        )

metrics_cb = GraphMetricsCallback()
# ======================
# TRAIN GCN
# Only TRAIN labels are used
# Validation labels are only used for validation
# Test labels are not used for training
# ======================
history = gcn.fit(
    [X_std, A_norm],
    Y,
    sample_weight=mask_tr.astype(np.float32),
    validation_data=(
        [X_std, A_norm],
        Y,
        mask_va.astype(np.float32)
    ),
    epochs=EPOCHS_GCN,
    batch_size=N,
    shuffle=False,
    verbose=1,
    callbacks=[
        metrics_cb,
        EarlyStopping(
            monitor="val_loss",
            patience=30,
            restore_best_weights=True
        )
    ]
)


# ======================
# FINAL PREDICTION
# ======================
pred_prob = gcn.predict(
    [X_std, A_norm],
    batch_size=N,
    verbose=0
)

pred = np.argmax(pred_prob, axis=1)

# ======================
# EVALUATION FUNCTION
# ======================
def evaluate_split(split_name, mask):
    print(f"\n===== {split_name} CLASSIFICATION REPORT =====")
    print(classification_report(
        labels[mask],
        pred[mask],
        target_names=class_names
    ))

    cm = confusion_matrix(labels[mask], pred[mask])

    plt.figure(figsize=(5, 5))
    ax = sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        square=True,
        linewidths=2,
        linecolor="white",
        cbar=False,
        xticklabels=class_names,
        yticklabels=class_names,
        annot_kws={"size": 10, "color": "red"}
    )

    ax.set_title(f"{split_name} Confusion Matrix", fontsize=12, weight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

    plt.xticks(rotation=45, fontsize=8)
    plt.yticks(rotation=45, fontsize=8)
    plt.tight_layout()
    plt.show()

    y_true_bin = label_binarize(
        labels[mask],
        classes=np.arange(num_classes)
    )

    y_score = pred_prob[mask]

    plt.figure(figsize=(5, 5))

    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(
            fpr,
            tpr,
            label=f"{class_names[i]} AUC = {roc_auc:.3f}"
        )

    plt.plot([0, 1], [0, 1], "k--")
    plt.title(f"{split_name} ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


# ======================
# VALIDATION RESULTS
# ======================
evaluate_split("Validation", mask_va)


# ======================
# TEST RESULTS
# ======================
evaluate_split("Test", mask_te)


# ======================
# ACCURACY CURVE
# ======================
epochs = range(1, len(metrics_cb.train_acc) + 1)

plt.figure(figsize=(5, 4))
plt.plot(epochs, metrics_cb.train_acc, label="Train Accuracy")
plt.plot(epochs, metrics_cb.val_acc, label="Validation Accuracy")
plt.plot(epochs, metrics_cb.test_acc, label="Test Accuracy")
plt.title("GCN Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()


# ======================
# LOSS CURVE
# ======================
plt.figure(figsize=(5, 4))
plt.plot(epochs, metrics_cb.train_loss, label="Train Loss")
plt.plot(epochs, metrics_cb.val_loss, label="Validation Loss")
plt.plot(epochs, metrics_cb.test_loss, label="Test Loss")
plt.title("GCN Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FULL COMPUTATIONAL METRICS + DESCRIPTIVE STATS FOR RBF GCN
# ============================================================

import os
import gc
import time
import psutil
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy import stats

# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "RBF-CNN-GCN Framework"

# Use the RBF GCN trained model
model_to_measure = gcn  # RBF-based GCN

# Input shape for GCN: number of features per node
input_shape = (X_std.shape[1],)
batch_size = N  # full-batch GCN

# ============================================================
# MEMORY HELPERS
# ============================================================

def get_memory_usage_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

def get_model_params(model):
    return model.count_params()

def get_model_size_mb(model):
    return get_model_params(model) * 4 / (1024 * 1024)

# ============================================================
# FLOPs ESTIMATION HELPERS
# ============================================================

def safe_shape(tensor):
    try:
        return tensor.shape.as_list()
    except Exception:
        try:
            return list(tensor.shape)
        except Exception:
            return None

def calculate_flops_per_image(model):
    total_flops = 0
    for layer in model.layers:
        try:
            if isinstance(layer, tf.keras.Model):
                total_flops += calculate_flops_per_image(layer)
                continue

            layer_flops = 0
            if isinstance(layer, tf.keras.layers.Conv2D):
                out_shape = safe_shape(layer.output)
                kernel_h, kernel_w = layer.kernel_size
                in_channels = int(layer.kernel.shape[-2])
                out_channels = int(layer.kernel.shape[-1])
                if out_shape is not None and len(out_shape) == 4:
                    out_h = out_shape[1]
                    out_w = out_shape[2]
                    if out_h is not None and out_w is not None:
                        layer_flops = 2 * out_h * out_w * kernel_h * kernel_w * in_channels * out_channels

            elif isinstance(layer, tf.keras.layers.DepthwiseConv2D):
                out_shape = safe_shape(layer.output)
                kernel_h, kernel_w = layer.kernel_size
                in_channels = int(layer.depthwise_kernel.shape[-2])
                depth_multiplier = int(layer.depthwise_kernel.shape[-1])
                if out_shape is not None and len(out_shape) == 4:
                    out_h = out_shape[1]
                    out_w = out_shape[2]
                    if out_h is not None and out_w is not None:
                        layer_flops = 2 * out_h * out_w * kernel_h * kernel_w * in_channels * depth_multiplier

            elif isinstance(layer, tf.keras.layers.Dense):
                kernel_shape = layer.kernel.shape
                if kernel_shape is not None:
                    in_features = int(kernel_shape[0])
                    out_features = int(kernel_shape[1])
                    layer_flops = 2 * in_features * out_features

            elif isinstance(layer, tf.keras.layers.BatchNormalization):
                out_shape = safe_shape(layer.output)
                if out_shape is not None:
                    units = np.prod([d for d in out_shape[1:] if d is not None])
                    layer_flops = 2 * units

            total_flops += layer_flops

        except Exception:
            continue

    return int(total_flops)

# ============================================================
# INFERENCE METRICS
# ============================================================

def measure_inference_metrics_gcn(model, X, A, warmup_runs=3, measured_runs=10):
    """
    Measures computational metrics for a GCN that takes features X and adjacency A.
    """
    if not hasattr(model, "predict"):
        raise TypeError("model must be a trained Keras model.")

    # Convert adjacency to dense for dummy runs if needed
    if hasattr(A, "todense") or hasattr(A, "toarray"):
        A_dense = np.array(A.todense(), dtype=np.float32)
    else:
        A_dense = np.array(A, dtype=np.float32)

    X_dummy = np.array(X, dtype=np.float32)

    # Warm-up
    for _ in range(warmup_runs):
        _ = model.predict([X_dummy, A_dense], batch_size=N, verbose=0)

    gc.collect()
    start_mem = get_memory_usage_mb()
    start_time = time.time()

    for _ in range(measured_runs):
        _ = model.predict([X_dummy, A_dense], batch_size=N, verbose=0)

    end_time = time.time()
    end_mem = get_memory_usage_mb()

    total_time = end_time - start_time
    latency_ms_per_img = total_time * 1000.0 / (len(X_dummy) * measured_runs)

    # FLOPs approximation uses CNN backbone only (optional)
    flops_per_image = calculate_flops_per_image(model)
    gflops_per_image = flops_per_image / 1e9
    flops_per_batch = flops_per_image * len(X_dummy)
    gflops_per_batch = flops_per_batch / 1e9

    params = get_model_params(model)
    size_mb = get_model_size_mb(model)

    return {
        "Params": int(params),
        "Params (M)": float(params / 1e6),
        "Model Size (MB)": float(size_mb),
        "Inference Latency (ms/img)": float(latency_ms_per_img),
        "FLOPs/Image (G)": float(gflops_per_image),
        "FLOPs/Batch (G)": float(gflops_per_batch),
        "Inference Memory Delta (MB)": float(end_mem - start_mem),
    }
# ============================================================
# DESCRIPTIVE STATISTICS HELPERS
# ============================================================

def descriptive_statistics(data):
    data = np.array(data, dtype=float)
    if data.size == 0: return (np.nan, np.nan, np.nan, np.nan, np.nan)
    mean = np.mean(data)
    std = np.std(data, ddof=1) if len(data) > 1 else 0.0
    median = np.median(data)
    iqr = np.percentile(data, 75) - np.percentile(data, 25)
    rng = np.ptp(data)
    return float(mean), float(std), float(median), float(iqr), float(rng)

def confidence_interval(data):
    data = np.array(data, dtype=float)
    if len(data) < 2: return (np.nan, np.nan)
    mean = np.mean(data)
    sem = stats.sem(data)
    if sem == 0 or np.isnan(sem): return (float(mean), float(mean))
    low, high = stats.t.interval(0.95, len(data)-1, loc=mean, scale=sem)
    return float(low), float(high)

def one_sample_ttest(data, baseline=0):
    data = np.array(data, dtype=float)
    if len(data) < 2: return (np.nan, np.nan)
    t, p = stats.ttest_1samp(data, baseline)
    return float(t), float(p)

def effect_size(data, baseline=0):
    data = np.array(data, dtype=float)
    if len(data) < 2: return (np.nan, np.nan)
    mean = np.mean(data)
    std = np.std(data, ddof=1)
    cohen_d = (mean - baseline) / std if std > 0 else np.nan
    n = len(data)
    hedges_g = cohen_d * (1 - (3 / (4 * n - 9))) if n > 2 else cohen_d
    return float(cohen_d), float(hedges_g)

def normality_test(data):
    data = np.array(data, dtype=float)
    if len(data) < 3: return (np.nan, np.nan)
    W, p = stats.shapiro(data)
    return float(W), float(p)

def wilcoxon_test(data, baseline=0):
    data = np.array(data, dtype=float)
    if len(data) < 3: return (np.nan, np.nan)
    try:
        stat, p = stats.wilcoxon(data - baseline)
        return float(stat), float(p)
    except: return (np.nan, np.nan)

def correlation(acc, loss):
    acc = np.array(acc, dtype=float)
    loss = np.array(loss, dtype=float)
    if len(acc) < 2 or len(loss) < 2: return (np.nan, np.nan, np.nan, np.nan)
    min_len = min(len(acc), len(loss))
    acc, loss = acc[:min_len], loss[:min_len]
    try: pearson_r, pearson_p = stats.pearsonr(acc, loss)
    except: pearson_r, pearson_p = np.nan, np.nan
    try: spearman_rho, spearman_p = stats.spearmanr(acc, loss)
    except: spearman_rho, spearman_p = np.nan, np.nan
    return float(pearson_r), float(pearson_p), float(spearman_rho), float(spearman_p)

# ============================================================
# USE TRAINING CALLBACK METRICS
# ============================================================

train_acc_final = metrics_cb.train_acc
train_loss_final = metrics_cb.train_loss
val_acc_final = metrics_cb.val_acc
val_loss_final = metrics_cb.val_loss

# ============================================================
# COMPUTE INFERENCE METRICS
# ============================================================


inference_metrics = measure_inference_metrics_gcn(
    model=gcn,
    X=X_std,
    A=A_norm,  # normalized adjacency matrix from RBF graph
    warmup_runs=3,
    measured_runs=10
)

# ============================================================
# COMPUTE STATISTICS
# ============================================================

acc_mean, acc_std, acc_median, acc_iqr, acc_range = descriptive_statistics(train_acc_final)
loss_mean, loss_std, loss_median, loss_iqr, loss_range = descriptive_statistics(train_loss_final)

acc_ci = confidence_interval(train_acc_final)
loss_ci = confidence_interval(train_loss_final)

acc_t, acc_p = one_sample_ttest(train_acc_final)
loss_t, loss_p = one_sample_ttest(train_loss_final)

acc_d, acc_g = effect_size(train_acc_final)
loss_d, loss_g = effect_size(train_loss_final)

acc_W, acc_Wp = normality_test(train_acc_final)
loss_W, loss_Wp = normality_test(train_loss_final)

acc_wilc, acc_wilc_p = wilcoxon_test(train_acc_final)
loss_wilc, loss_wilc_p = wilcoxon_test(train_loss_final)

pearson_r, pearson_p, spearman_rho, spearman_p = correlation(train_acc_final, train_loss_final)

val_acc_mean, val_acc_std, val_acc_median, val_acc_iqr, val_acc_range = descriptive_statistics(val_acc_final)
val_loss_mean, val_loss_std, val_loss_median, val_loss_iqr, val_loss_range = descriptive_statistics(val_loss_final)



# ============================================================
# PRINT METRICS VERTICALLY
# ============================================================

print("\n--- RBF GCN Computational Metrics ---")
for k, v in inference_metrics.items():
    print(f"{k}: {v}")

print("\n--- Training Metrics ---")
for k, v in compute_metrics["Train Stats"].items():
    print(f"{k}: {v}")

print("\n--- Validation Metrics ---")
for k, v in compute_metrics["Validation Stats"].items():
    print(f"{k}: {v}")

In [ ]:
# ============================================================
# FULL RBF-GCN COMPUTATIONAL METRICS + DESCRIPTIVE STATS
# ============================================================

import os, time, gc, psutil
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy import stats
from scipy.sparse import coo_matrix
from sklearn.preprocessing import label_binarize
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss
from spektral.layers import GCNConv
from tensorflow.keras import layers, Model, Input, regularizers

# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "RBF-GCN Framework"
EPOCHS_GCN = 50
DROPOUT = 0.3
SEED = 123

np.random.seed(SEED)
tf.random.set_seed(SEED)

# X_std: CNN feature matrix
# labels: integer labels
# mask_tr, mask_va, mask_te: boolean arrays for train/val/test
# num_classes: number of classes

# ============================================================
# MEMORY HELPERS
# ============================================================

def get_memory_usage_mb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024**2)

# ============================================================
# BUILD RBF GRAPH
# ============================================================

def build_rbf_graph(X_std, k=12, gamma=None):
    N, F = X_std.shape
    if gamma is None:
        gamma = 1.0 / max(1, F)
    from sklearn.neighbors import NearestNeighbors
    rows, cols, data = [], [], []
    nn = NearestNeighbors(n_neighbors=k+1).fit(X_std)
    dist, knn_idx = nn.kneighbors(X_std)
    for i in range(N):
        for j, d in zip(knn_idx[i], dist[i]):
            if i==j: continue
            w = np.exp(-gamma*d*d)
            if w <= 0: continue
            rows.append(i)
            cols.append(j)
            data.append(w)
    A_dir = coo_matrix((data, (rows, cols)), shape=(N, N), dtype=np.float32)
    A = A_dir.maximum(A_dir.T).tocsr()
    A.setdiag(1.0)
    from spektral.utils.convolution import gcn_filter
    A_norm = gcn_filter(A)
    
    degrees = np.array(A.sum(axis=1)).flatten() - 1
    graph_stats = {
        "Edges": (A.nnz - N)//2,
        "Avg Degree": degrees.mean(),
        "Min Degree": degrees.min(),
        "Max Degree": degrees.max(),
        "Memory Usage MB": get_memory_usage_mb()
    }
    return A, A_norm, graph_stats

# ============================================================
# BUILD GCN
# ============================================================

def build_gcn(F, N_nodes, num_classes, dropout=DROPOUT, lr=0.005):
    X_in = Input(shape=(F,))
    A_in = Input(shape=(N_nodes, N_nodes), sparse=True)
    h = GCNConv(64, activation="relu", kernel_regularizer=regularizers.l2(5e-4))([X_in, A_in])
    h = layers.Dropout(dropout)(h)
    out = GCNConv(num_classes, activation="softmax")([h, A_in])
    model = Model([X_in, A_in], out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)
    )
    return model

# ============================================================
# STATISTICAL METRICS
# ============================================================

def compute_stats(pred_prob, true_labels, mask, num_classes):
    pred_labels = np.argmax(pred_prob, axis=1)
    stats_dict = {}
    stats_dict['accuracy'] = accuracy_score(true_labels[mask], pred_labels[mask])
    stats_dict['precision'] = precision_score(true_labels[mask], pred_labels[mask], average='weighted', zero_division=0)
    stats_dict['recall'] = recall_score(true_labels[mask], pred_labels[mask], average='weighted', zero_division=0)
    stats_dict['f1'] = f1_score(true_labels[mask], pred_labels[mask], average='weighted', zero_division=0)
    y_true_bin = label_binarize(true_labels[mask], classes=np.arange(num_classes))
    stats_dict['log_loss'] = log_loss(y_true_bin, pred_prob[mask])
    return stats_dict

# ============================================================
# TRAIN GCN WITH RBF GRAPH
# ============================================================

def train_rbf_gcn(X_std, labels, mask_tr, mask_va, mask_te, num_classes, k=12, epochs=EPOCHS_GCN):
    N, F = X_std.shape
    A, A_norm, graph_stats = build_rbf_graph(X_std, k=k)
    Y = tf.keras.utils.to_categorical(labels, num_classes).astype(np.float32)
    gcn = build_gcn(F, N, num_classes)
    
    epoch_times = []
    train_acc_list, val_acc_list = [], []
    
    for epoch in range(epochs):
        start = time.time()
        gcn.fit([X_std, A_norm], Y,
                sample_weight=mask_tr.astype(np.float32),
                validation_data=([X_std, A_norm], Y, mask_va.astype(np.float32)),
                epochs=1, batch_size=N, shuffle=False, verbose=0)
        epoch_times.append(time.time() - start)
        
        pred_prob = gcn.predict([X_std, A_norm], batch_size=N, verbose=0)
        train_acc_list.append(compute_stats(pred_prob, labels, mask_tr, num_classes)['accuracy'])
        val_acc_list.append(compute_stats(pred_prob, labels, mask_va, num_classes)['accuracy'])
    
    pred_prob_final = gcn.predict([X_std, A_norm], batch_size=N, verbose=0)
    train_stats = compute_stats(pred_prob_final, labels, mask_tr, num_classes)
    val_stats = compute_stats(pred_prob_final, labels, mask_va, num_classes)
    test_stats = compute_stats(pred_prob_final, labels, mask_te, num_classes)
    
    return gcn, graph_stats, epoch_times, train_stats, val_stats, test_stats, train_acc_list, val_acc_list

# ============================================================
# RUN RBF-GCN
# ============================================================

gcn_model, graph_stats, epoch_times, train_stats, val_stats, test_stats, train_acc_list, val_acc_list = train_rbf_gcn(
    X_std, labels, mask_tr, mask_va, mask_te, num_classes, k=12, epochs=50
)

# ============================================================
# COMBINE METRICS INTO DATAFRAME
# ============================================================

compute_metrics_df = pd.DataFrame({
    "Model": [MODEL_NAME],
    "Edges": [graph_stats["Edges"]],
    "Avg Degree": [graph_stats["Avg Degree"]],
    "Min Degree": [graph_stats["Min Degree"]],
    "Max Degree": [graph_stats["Max Degree"]],
    "Memory MB": [graph_stats["Memory Usage MB"]],
    "Total Training Time (s)": [sum(epoch_times)],
    "Train Accuracy": [train_stats['accuracy']],
    "Val Accuracy": [val_stats['accuracy']],
    "Test Accuracy": [test_stats['accuracy']],
    "Train F1": [train_stats['f1']],
    "Val F1": [val_stats['f1']],
    "Test F1": [test_stats['f1']]
})

display(compute_metrics_df)
compute_metrics_df.to_csv("rbf_gcn_computational_cost_vs_performance.csv", index=False)
print("\nSaved as: rbf_gcn_computational_cost_vs_performance.csv")

# ============================================================
# OPTIONAL: Plot Computational Cost vs Accuracy
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.plot(epoch_times, train_acc_list, label="Train Acc")
plt.plot(epoch_times, val_acc_list, label="Val Acc")
plt.xlabel("Epoch Time (s)")
plt.ylabel("Accuracy")
plt.title("RBF-GCN: Computational Cost vs Performance")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# FULL RBF-GCN COMPUTATIONAL COST + DESCRIPTIVE STATISTICS
# ============================================================

import os, time, gc, psutil
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy import stats
from scipy.sparse import coo_matrix
from sklearn.preprocessing import label_binarize
from spektral.layers import GCNConv
from tensorflow.keras import layers, Model, Input, regularizers
import matplotlib.pyplot as plt

# ================================
# CONFIG
# ================================
MODEL_NAME = "RBF-GCN Framework"
BATCH_SIZE = 32
EPOCHS_GCN = 50
DROPOUT = 0.3
SEED = 123

np.random.seed(SEED)
tf.random.set_seed(SEED)

# X_std: CNN features already extracted
# labels: numpy array of labels
# mask_tr, mask_va, mask_te: boolean arrays for train/val/test split
# num_classes: int

# ================================
# MEMORY HELPER
# ================================
def get_memory_usage_mb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024**2)

# ================================
# COMPUTE GRAPH (RBF) 
# ================================
def build_rbf_graph(X_std, k=12, gamma=None):
    N = X_std.shape[0]
    F = X_std.shape[1]
    if gamma is None:
        gamma = 1.0 / max(1, F)
    
    from sklearn.neighbors import NearestNeighbors
    rows, cols, data = [], [], []
    nn = NearestNeighbors(n_neighbors=k+1, metric="euclidean").fit(X_std)
    dist, knn_idx = nn.kneighbors(X_std)
    for i in range(N):
        for j, d in zip(knn_idx[i], dist[i]):
            if i == j: continue
            w = np.exp(-gamma*d*d)
            if w <= 0: continue
            rows.append(i); cols.append(j); data.append(w)
    
    A_dir = coo_matrix((data, (rows, cols)), shape=(N, N), dtype=np.float32)
    A = A_dir.maximum(A_dir.T).tocsr()
    A.setdiag(1.0)
    
    # Normalize for GCN
    from spektral.utils.convolution import gcn_filter
    A_norm = gcn_filter(A)
    
    # Graph stats
    degrees = np.array(A.sum(axis=1)).flatten() - 1
    graph_stats = {
        "Edges": (A.nnz - N) // 2,
        "Avg Degree": degrees.mean(),
        "Min Degree": degrees.min(),
        "Max Degree": degrees.max(),
        "Memory Usage MB": get_memory_usage_mb()
    }
    
    return A, A_norm, graph_stats

# ================================
# BUILD GCN MODEL
# ================================
def build_gcn(F, N_nodes, num_classes, dropout=DROPOUT, lr=0.005):
    X_in = Input(shape=(F,))
    A_in = Input(shape=(N_nodes, N_nodes), sparse=True)
    h = GCNConv(64, activation="relu", kernel_regularizer=regularizers.l2(5e-4))([X_in, A_in])
    h = layers.Dropout(dropout)(h)
    out = GCNConv(num_classes, activation="softmax")([h, A_in])
    model = Model([X_in, A_in], out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        weighted_metrics=["accuracy"]
    )
    return model

# ================================
# STATISTICAL METRICS
# ================================
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss

def compute_stats(pred_prob, true_labels, mask, num_classes):
    pred_labels = np.argmax(pred_prob, axis=1)
    stats_dict = {}
    stats_dict['accuracy'] = accuracy_score(true_labels[mask], pred_labels[mask])
    stats_dict['precision'] = precision_score(true_labels[mask], pred_labels[mask], average='weighted', zero_division=0)
    stats_dict['recall'] = recall_score(true_labels[mask], pred_labels[mask], average='weighted', zero_division=0)
    stats_dict['f1'] = f1_score(true_labels[mask], pred_labels[mask], average='weighted', zero_division=0)
    # Log-loss using probabilities
    y_true_bin = label_binarize(true_labels[mask], classes=np.arange(num_classes))
    stats_dict['log_loss'] = log_loss(y_true_bin, pred_prob[mask])
    return stats_dict
# ================================
# TRAIN GCN AND TRACK COST
# ================================
def train_gcn_rbf(X_std, labels, mask_tr, mask_va, mask_te, num_classes, k=12, epochs=EPOCHS_GCN):
    N = X_std.shape[0]
    F = X_std.shape[1]
    
    # --- Build RBF graph ---
    A, A_norm, graph_stats = build_rbf_graph(X_std, k=k)
    
    # --- Labels ---
    Y = tf.keras.utils.to_categorical(labels, num_classes).astype(np.float32)
    
    # --- Build GCN ---
    gcn = build_gcn(F, N, num_classes)
    
    epoch_times = []
    train_acc_list, val_acc_list = [], []
    
    for epoch in range(epochs):
        start = time.time()
        gcn.fit(
            [X_std, A_norm], Y,
            sample_weight=mask_tr.astype(np.float32),
            validation_data=([X_std, A_norm], Y, mask_va.astype(np.float32)),
            epochs=1, batch_size=N, shuffle=False, verbose=0
        )
        epoch_times.append(time.time() - start)
        
        pred_prob = gcn.predict([X_std, A_norm], batch_size=N, verbose=0)
        train_acc_list.append(compute_stats(pred_prob, labels, mask_tr, num_classes)['accuracy'])
        val_acc_list.append(compute_stats(pred_prob, labels, mask_va, num_classes)['accuracy'])
    
    # --- Test metrics ---
    pred_prob_final = gcn.predict([X_std, A_norm], batch_size=N, verbose=0)
    train_stats = compute_stats(pred_prob_final, labels, mask_tr, num_classes)
    val_stats = compute_stats(pred_prob_final, labels, mask_va, num_classes)
    test_stats = compute_stats(pred_prob_final, labels, mask_te, num_classes)
    
    return gcn, graph_stats, epoch_times, train_stats, val_stats, test_stats, train_acc_list, val_acc_list

# ================================
# RUN
# ================================
gcn_model, graph_stats, epoch_times, train_stats, val_stats, test_stats, train_acc_list, val_acc_list = train_gcn_rbf(
    X_std, labels, mask_tr, mask_va, mask_te, num_classes, k=12, epochs=50
)

# ================================
# COMPUTATIONAL COST + PERFORMANCE DF
# ================================
compute_metrics_df = pd.DataFrame({
    "Model": [MODEL_NAME],
    "Edges": [graph_stats["Edges"]],
    "Avg Degree": [graph_stats["Avg Degree"]],
    "Min Degree": [graph_stats["Min Degree"]],
    "Max Degree": [graph_stats["Max Degree"]],
    "Memory MB": [graph_stats["Memory Usage MB"]],
    "Total Training Time (s)": [sum(epoch_times)],
    "Train Accuracy": [train_stats['accuracy']],
    "Val Accuracy": [val_stats['accuracy']],
    "Test Accuracy": [test_stats['accuracy']],
    "Train F1": [train_stats['f1']],
    "Val F1": [val_stats['f1']],
    "Test F1": [test_stats['f1']]
})

display(compute_metrics_df)
compute_metrics_df.to_csv("rbf_gcn_computational_cost_vs_performance.csv", index=False)
print("\nSaved as: rbf_gcn_computational_cost_vs_performance.csv")

# ================================
# OPTIONAL PLOTS
# ================================
plt.figure(figsize=(6,4))
plt.plot(epoch_times, train_acc_list, label="Train Acc")
plt.plot(epoch_times, val_acc_list, label="Val Acc")
plt.xlabel("Epoch Time (s)")
plt.ylabel("Accuracy")
plt.title("RBF-GCN: Computational Cost vs Performance")
plt.legend()
plt.tight_layout()
plt.show()


***
<a name='import Packages'>
    
# 4 <span style='color:blue'>|</span> Domain 

In [ ]:
import gc
import os
import psutil

# Function to check memory usage
def memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    print(f"Memory usage: {mem_info.rss / 1024**2:.2f} MB")

# Function to clear TensorFlow cache (if using TensorFlow)
def clear_tensorflow_cache():
    try:
        import tensorflow as tf
        print("Clearing TensorFlow cache...")
        tf.keras.backend.clear_session()
    except ImportError:
        print("TensorFlow is not installed.")

# Function to clear PyTorch cache (if using PyTorch)
def clear_pytorch_cache():
    try:
        import torch
        print("Clearing PyTorch cache...")
        torch.cuda.empty_cache()
    except ImportError:
        print("PyTorch is not installed.")

# Force garbage collection
def clear_memory():
    print("Clearing memory and garbage collection...")
    gc.collect()

# Main function to clear cache and memory
def clear_cache_and_memory():
    print("Before clearing:")
    memory_usage()

    clear_tensorflow_cache()
    clear_pytorch_cache()
    clear_memory()

    print("After clearing:")
    memory_usage()

# Example usage
if __name__ == "__main__":
    clear_cache_and_memory()


In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    accuracy_score,
    log_loss
)
from sklearn.manifold import TSNE

import tensorflow as tf
from tensorflow.keras import layers, models, Input, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from spektral.layers import GCNConv
from spektral.utils.convolution import gcn_filter
# ======================
# CONFIG
# ======================
data_dir = r"D:/GCN/Brain_Tumor/four_class"

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 123

AUGMENT = True
LR_CNN = 1e-3
EPOCHS_CNN = 30

K_DEFAULT = 12
USE_PCA = True
PCA_DIM = 256

LR_GCN = 0.005
EPOCHS_GCN = 200
DROPOUT = 0.3

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ======================
# LOAD DATA
# ======================
class_names = sorted([
    d for d in os.listdir(data_dir)
    if os.path.isdir(os.path.join(data_dir, d))
])

class_to_idx = {c: i for i, c in enumerate(class_names)}

paths, labels = [], []

for c in class_names:
    for p in glob.glob(os.path.join(data_dir, c, "*")):
        if p.lower().endswith((".jpg", ".png", ".jpeg", ".bmp", ".tif", ".tiff")):
            paths.append(p)
            labels.append(class_to_idx[c])

paths = np.array(paths)
labels = np.array(labels, dtype=np.int32)

N = len(paths)
num_classes = len(class_names)

print("Images:", N)
print("Classes:", num_classes)
print("Class names:", class_names)


# ======================
# SPLIT: 70 TRAIN, 20 VAL, 10 TEST
# ======================
idx = np.arange(N)

idx_temp, idx_te = train_test_split(
    idx,
    test_size=0.10,
    random_state=SEED,
    stratify=labels
)

idx_tr, idx_va = train_test_split(
    idx_temp,
    test_size=0.2222,
    random_state=SEED,
    stratify=labels[idx_temp]
)

mask_tr = np.zeros(N, dtype=bool)
mask_va = np.zeros(N, dtype=bool)
mask_te = np.zeros(N, dtype=bool)

mask_tr[idx_tr] = True
mask_va[idx_va] = True
mask_te[idx_te] = True

print("Train:", len(idx_tr))
print("Validation:", len(idx_va))
print("Test:", len(idx_te))


# ======================
# DATA PIPELINE
# ======================
def load_img(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 3])
    return img, label


def augment_img(img, label):
    if AUGMENT:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        k = tf.random.uniform([], 0, 4, dtype=tf.int32)
        img = tf.image.rot90(img, k)
    return img, label


def make_ds(idxs, training=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths[idxs], labels[idxs]))

    if shuffle:
        ds = ds.shuffle(len(idxs), seed=SEED)

    ds = ds.map(load_img, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.map(augment_img, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_ds(idx_tr, training=True, shuffle=True)
val_ds = make_ds(idx_va, training=False, shuffle=False)
all_ds = make_ds(idx, training=False, shuffle=False)


# ======================
# CNN FEATURE EXTRACTOR
# CNN is used only to learn image features
# ======================
def build_cnn():
    inp = Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    feat = layers.Dense(512, activation="relu", name="feat")(x)
    out = layers.Dropout(0.5)(feat)
    out = layers.Dense(num_classes, activation="softmax")(out)

    cnn = Model(inp, out)
    backbone = Model(inp, feat)

    return cnn, backbone


cnn, backbone = build_cnn()

cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_CNN),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_CNN,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True
        )
    ],
    verbose=1
)


# ======================
# EXTRACT CNN FEATURES
# ======================
def extract_features(ds):
    X, Y = [], []

    for xb, yb in ds:
        feat = backbone(xb, training=False).numpy()
        X.append(feat)
        Y.append(yb.numpy())

    return np.vstack(X), np.concatenate(Y)


X_all, y_all = extract_features(all_ds)

print("CNN feature shape:", X_all.shape)


# ======================
# PCA + STANDARDIZATION
# Important: fit PCA and scaler only on TRAIN data
# ======================
X_tr = X_all[idx_tr]
X_va = X_all[idx_va]
X_te = X_all[idx_te]

if USE_PCA:
    pca_dim = min(PCA_DIM, X_tr.shape[1])
    pca = PCA(n_components=pca_dim, random_state=SEED)
    X_tr = pca.fit_transform(X_tr)
    X_va = pca.transform(X_va)
    X_te = pca.transform(X_te)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_va = scaler.transform(X_va)
X_te = scaler.transform(X_te)

F = X_tr.shape[1]

X_std = np.zeros((N, F), dtype=np.float32)
X_std[idx_tr] = X_tr
X_std[idx_va] = X_va
X_std[idx_te] = X_te

print("Final feature shape for graph:", X_std.shape)

# ======================
# DOMAIN-RULES GRAPH BUILDING
# Domain graph builds edges using folder/class information
# ======================

def build_graph_domain_rules(paths_list, class_names):
    N_local = len(paths_list)

    class_to_indices = {c: [] for c in class_names}

    for i, p in enumerate(paths_list):
        cls = os.path.basename(os.path.dirname(p))

        if cls in class_to_indices:
            class_to_indices[cls].append(i)

    rows, cols, data = [], [], []

    for idxs in class_to_indices.values():
        for i in idxs:
            for j in idxs:
                rows.append(i)
                cols.append(j)
                data.append(1.0)

    A = coo_matrix(
        (data, (rows, cols)),
        shape=(N_local, N_local),
        dtype=np.float32
    ).tocsr()

    A.setdiag(1.0)

    return A


A = build_graph_domain_rules(
    paths_list=paths,
    class_names=class_names
)

# Normalize for GCN
A_norm = gcn_filter(A)



print("\n===== GRAPH STATS =====")
nnz_total = A.nnz
self_loops = N
undirected_edges = (nnz_total - self_loops) // 2
degrees = np.array(A.sum(axis=1)).flatten() - 1

print("Edges:", undirected_edges)
print("Avg degree:", degrees.mean())
print("Min degree:", degrees.min())
print("Max degree:", degrees.max())

n_comp, labels_comp = connected_components(A, directed=False)
print("Components:", n_comp)


# ======================
# OPTIONAL t-SNE VISUALIZATION
# ======================
print("[t-SNE] computing...")

sample_N = min(3000, N)
sample_idx = np.random.choice(N, sample_N, replace=False)

X_ts = X_std[sample_idx]
y_ts = labels[sample_idx]

if X_ts.shape[1] > 50:
    X_ts = PCA(50, random_state=SEED).fit_transform(X_ts)

X_2d = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=SEED
).fit_transform(X_ts)

plt.figure(figsize=(7, 6))
for i, c in enumerate(class_names):
    m = y_ts == i
    plt.scatter(X_2d[m, 0], X_2d[m, 1], s=8, label=c)

plt.title("t-SNE of CNN Features")
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(figsize=(5, 4))
plt.hist(degrees, bins=40, edgecolor="black")
plt.title("Graph Degree Distribution")
plt.xlabel("Degree")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


# ======================
# GCN CLASSIFIER
# GCN performs final classification
# ======================
def build_gcn(F):
    X_in = Input(shape=(F,))
    A_in = Input(shape=(N, N), sparse=True)

    h = GCNConv(
        64,
        activation="relu",
        kernel_regularizer=regularizers.l2(5e-4)
    )([X_in, A_in])

    h = layers.Dropout(DROPOUT)(h)

    out = GCNConv(
        num_classes,
        activation="softmax"
    )([h, A_in])

    model = Model([X_in, A_in], out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_GCN),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        weighted_metrics=["accuracy"]
    )
    return model
Y = to_categorical(labels, num_classes).astype(np.float32)
gcn = build_gcn(X_std.shape[1])

# ======================
# CALLBACK FOR GCN CURVES
# ======================
class GraphMetricsCallback(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()

        self.train_acc = []
        self.val_acc = []
        self.test_acc = []

        self.train_loss = []
        self.val_loss = []
        self.test_loss = []

    def on_epoch_end(self, epoch, logs=None):
        pred_prob_epoch = self.model.predict(
            [X_std, A_norm],
            batch_size=N,
            verbose=0
        )

        pred_epoch = np.argmax(pred_prob_epoch, axis=1)

        self.train_acc.append(
            accuracy_score(labels[mask_tr], pred_epoch[mask_tr])
        )
        self.val_acc.append(
            accuracy_score(labels[mask_va], pred_epoch[mask_va])
        )
        self.test_acc.append(
            accuracy_score(labels[mask_te], pred_epoch[mask_te])
        )

        self.train_loss.append(
            log_loss(
                labels[mask_tr],
                pred_prob_epoch[mask_tr],
                labels=np.arange(num_classes)
            )
        )
        self.val_loss.append(
            log_loss(
                labels[mask_va],
                pred_prob_epoch[mask_va],
                labels=np.arange(num_classes)
            )
        )
        self.test_loss.append(
            log_loss(
                labels[mask_te],
                pred_prob_epoch[mask_te],
                labels=np.arange(num_classes)
            )
        )

        print(
            f" | custom_train_acc: {self.train_acc[-1]:.4f}"
            f" | custom_val_acc: {self.val_acc[-1]:.4f}"
            f" | custom_test_acc: {self.test_acc[-1]:.4f}"
        )

metrics_cb = GraphMetricsCallback()
# ======================
# TRAIN GCN
# Only TRAIN labels are used
# Validation labels are only used for validation
# Test labels are not used for training
# ======================
history = gcn.fit(
    [X_std, A_norm],
    Y,
    sample_weight=mask_tr.astype(np.float32),
    validation_data=(
        [X_std, A_norm],
        Y,
        mask_va.astype(np.float32)
    ),
    epochs=EPOCHS_GCN,
    batch_size=N,
    shuffle=False,
    verbose=1,
    callbacks=[
        metrics_cb,
        EarlyStopping(
            monitor="val_loss",
            patience=30,
            restore_best_weights=True
        )
    ]
)


# ======================
# FINAL PREDICTION
# ======================
pred_prob = gcn.predict(
    [X_std, A_norm],
    batch_size=N,
    verbose=0
)

pred = np.argmax(pred_prob, axis=1)

# ======================
# EVALUATION FUNCTION
# ======================
def evaluate_split(split_name, mask):
    print(f"\n===== {split_name} CLASSIFICATION REPORT =====")
    print(classification_report(
        labels[mask],
        pred[mask],
        target_names=class_names
    ))

    cm = confusion_matrix(labels[mask], pred[mask])

    plt.figure(figsize=(5, 5))
    ax = sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        square=True,
        linewidths=2,
        linecolor="white",
        cbar=False,
        xticklabels=class_names,
        yticklabels=class_names,
        annot_kws={"size": 10, "color": "red"}
    )

    ax.set_title(f"{split_name} Confusion Matrix", fontsize=12, weight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

    plt.xticks(rotation=45, fontsize=8)
    plt.yticks(rotation=45, fontsize=8)
    plt.tight_layout()
    plt.show()

    y_true_bin = label_binarize(
        labels[mask],
        classes=np.arange(num_classes)
    )

    y_score = pred_prob[mask]

    plt.figure(figsize=(5, 5))

    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(
            fpr,
            tpr,
            label=f"{class_names[i]} AUC = {roc_auc:.3f}"
        )

    plt.plot([0, 1], [0, 1], "k--")
    plt.title(f"{split_name} ROC Curve")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


# ======================
# VALIDATION RESULTS
# ======================
evaluate_split("Validation", mask_va)


# ======================
# TEST RESULTS
# ======================
evaluate_split("Test", mask_te)


# ======================
# ACCURACY CURVE
# ======================
epochs = range(1, len(metrics_cb.train_acc) + 1)

plt.figure(figsize=(5, 4))
plt.plot(epochs, metrics_cb.train_acc, label="Train Accuracy")
plt.plot(epochs, metrics_cb.val_acc, label="Validation Accuracy")
plt.plot(epochs, metrics_cb.test_acc, label="Test Accuracy")
plt.title("GCN Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()


# ======================
# LOSS CURVE
# ======================
plt.figure(figsize=(5, 4))
plt.plot(epochs, metrics_cb.train_loss, label="Train Loss")
plt.plot(epochs, metrics_cb.val_loss, label="Validation Loss")
plt.plot(epochs, metrics_cb.test_loss, label="Test Loss")
plt.title("GCN Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import os
import gc
import time
import psutil
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy import stats

# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "CNN-GCN Framework"
model_to_measure = cnn   # <-- your trained Keras model

input_shape = (150, 150, 3)
batch_size = 32

# Set correct random baseline
NUM_CLASSES = 4
RANDOM_BASELINE = 1 / NUM_CLASSES   # 0.25 for 4-class

# ============================================================
# DEVICE INFO
# ============================================================

gpus = tf.config.list_physical_devices('GPU')
DEVICE_USED = "GPU" if gpus else "CPU"

# ============================================================
# MEMORY HELPERS
# ============================================================

def get_memory_usage_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

def get_model_params(model):
    return model.count_params()

def get_model_size_mb(model):
    return get_model_params(model) * 4 / (1024 * 1024)

# ============================================================
# FLOPs ESTIMATION
# ============================================================

def safe_shape(tensor):
    try:
        return tensor.shape.as_list()
    except:
        try:
            return list(tensor.shape)
        except:
            return None

def calculate_flops_per_image(model):
    total_flops = 0

    for layer in model.layers:

        if isinstance(layer, tf.keras.Model):
            total_flops += calculate_flops_per_image(layer)
            continue

        layer_flops = 0

        if isinstance(layer, tf.keras.layers.Conv2D):
            out_shape = safe_shape(layer.output)
            if out_shape and len(out_shape) == 4:
                out_h, out_w = out_shape[1], out_shape[2]
                if out_h and out_w:
                    kh, kw = layer.kernel_size
                    in_ch = int(layer.kernel.shape[-2])
                    out_ch = int(layer.kernel.shape[-1])
                    layer_flops = 2 * out_h * out_w * kh * kw * in_ch * out_ch

        elif isinstance(layer, tf.keras.layers.DepthwiseConv2D):
            out_shape = safe_shape(layer.output)
            if out_shape and len(out_shape) == 4:
                out_h, out_w = out_shape[1], out_shape[2]
                if out_h and out_w:
                    kh, kw = layer.kernel_size
                    in_ch = int(layer.depthwise_kernel.shape[-2])
                    mult = int(layer.depthwise_kernel.shape[-1])
                    layer_flops = 2 * out_h * out_w * kh * kw * in_ch * mult

        elif isinstance(layer, tf.keras.layers.Dense):
            kernel_shape = layer.kernel.shape
            if kernel_shape:
                in_f = int(kernel_shape[0])
                out_f = int(kernel_shape[1])
                layer_flops = 2 * in_f * out_f

        elif isinstance(layer, tf.keras.layers.BatchNormalization):
            out_shape = safe_shape(layer.output)
            if out_shape:
                units = np.prod([d for d in out_shape[1:] if d])
                layer_flops = 2 * units

        total_flops += layer_flops

    return int(total_flops)

# ============================================================
# INFERENCE METRICS
# ============================================================

def measure_inference_metrics(model, batch_size, input_shape,
                              warmup_runs=5,
                              measured_runs=30):

    dummy_input = np.random.rand(batch_size, *input_shape).astype(np.float32)

    # Warmup
    for _ in range(warmup_runs):
        _ = model.predict(dummy_input, verbose=0)

    gc.collect()
    tf.keras.backend.clear_session()
    gc.collect()

    start_mem = get_memory_usage_mb()
    start_time = time.time()

    for _ in range(measured_runs):
        _ = model.predict(dummy_input, verbose=0)

    end_time = time.time()
    end_mem = get_memory_usage_mb()

    total_time = end_time - start_time

    latency_ms = total_time * 1000 / (batch_size * measured_runs)
    throughput = (batch_size * measured_runs) / total_time

    flops_img = calculate_flops_per_image(model)
    gflops_img = flops_img / 1e9

    return {
        "Params": get_model_params(model),
        "Params (M)": get_model_params(model) / 1e6,
        "Model Size (MB)": get_model_size_mb(model),
        "Inference Latency (ms/img)": latency_ms,
        "Throughput (img/sec)": throughput,
        "FLOPs/Image (G)": gflops_img,
        "FLOPs/Batch (G)": gflops_img * batch_size,
        "Inference Memory Delta (MB)": end_mem - start_mem,
    }

# ============================================================
# STATISTICS FUNCTIONS
# ============================================================

def descriptive_statistics(data):
    data = np.array(data, dtype=float)
    if len(data) == 0:
        return (np.nan,)*5
    return (
        np.mean(data),
        np.std(data, ddof=1) if len(data)>1 else 0,
        np.median(data),
        np.percentile(data,75)-np.percentile(data,25),
        np.ptp(data)
    )

def confidence_interval(data):
    data = np.array(data, dtype=float)
    if len(data)<2:
        return (np.nan,np.nan)
    mean=np.mean(data)
    sem=stats.sem(data)
    return stats.t.interval(0.95,len(data)-1,loc=mean,scale=sem)

def one_sample_ttest(data, baseline):
    if len(data)<2:
        return (np.nan,np.nan)
    return stats.ttest_1samp(data, baseline)

# ============================================================
# GET TRAINING CURVES
# ============================================================

if "metrics_cb" in globals():
    train_acc = metrics_cb.train_acc
    train_loss = metrics_cb.train_loss
    val_acc = getattr(metrics_cb,"val_acc",[])
    val_loss = getattr(metrics_cb,"val_loss",[])
else:
    train_acc = history.history.get("accuracy",[])
    train_loss = history.history.get("loss",[])
    val_acc = history.history.get("val_accuracy",[])
    val_loss = history.history.get("val_loss",[])

# Optional test stats if available
test_acc = globals().get("test_acc",[])
test_loss = globals().get("test_loss",[])

# ============================================================
# COMPUTE METRICS
# ============================================================

inference_metrics = measure_inference_metrics(
    model_to_measure,
    batch_size,
    input_shape
)

acc_mean, acc_std, acc_median, acc_iqr, acc_range = descriptive_statistics(train_acc)
loss_mean, loss_std, loss_median, loss_iqr, loss_range = descriptive_statistics(train_loss)

val_acc_mean, val_acc_std, _, _, _ = descriptive_statistics(val_acc)
val_loss_mean, val_loss_std, _, _, _ = descriptive_statistics(val_loss)

test_acc_mean, test_acc_std, _, _, _ = descriptive_statistics(test_acc)
test_loss_mean, test_loss_std, _, _, _ = descriptive_statistics(test_loss)

acc_ci = confidence_interval(train_acc)
loss_ci = confidence_interval(train_loss)

acc_t, acc_p = one_sample_ttest(train_acc, RANDOM_BASELINE)

# ============================================================
# DATAFRAME
# ============================================================

final_df = pd.DataFrame({
    "Model":[MODEL_NAME],
    "Device":[DEVICE_USED],
    **inference_metrics,
    "Train Acc Mean":[acc_mean],
    "Train Acc Std":[acc_std],
    "Val Acc Mean":[val_acc_mean],
    "Val Acc Std":[val_acc_std],
    "Test Acc Mean":[test_acc_mean],
    "Test Acc Std":[test_acc_std],
    "Train Loss Mean":[loss_mean],
    "Train Loss Std":[loss_std],
    "Val Loss Mean":[val_loss_mean],
    "Val Loss Std":[val_loss_std],
    "Test Loss Mean":[test_loss_mean],
    "Test Loss Std":[test_loss_std],
    "Train Acc 95% CI":[acc_ci],
    "Train Loss 95% CI":[loss_ci],
    "Train Acc T-test p-value":[acc_p]
})

final_df = final_df.round(4)

pd.set_option('display.max_columns',None)
display(final_df)

# ============================================================
# PRINT RESULTS
# ============================================================

print("\n--- Model Computational Metrics and Descriptive Stats ---")
for k, v in compute_metrics.items():
    print(f"{k}: {v}")


# ============================================================
# COMPUTATIONAL COST DATAFRAME
# ============================================================

compute_metrics_df = pd.DataFrame({
    "Model": [MODEL_NAME],
    "Params": [compute_metrics["Params"]],
    "Params (M)": [compute_metrics["Params (M)"]],
    "Model Size (MB)": [compute_metrics["Model Size (MB)"]],
    "Inference Latency (ms/img)": [compute_metrics["Inference Latency (ms/img)"]],
    "FLOPs/Image (G)": [compute_metrics["FLOPs/Image (G)"]],
    "FLOPs/Batch (G)": [compute_metrics["FLOPs/Batch (G)"]],
    "Inference Memory Delta (MB)": [compute_metrics["Inference Memory Delta (MB)"]],
})

compute_metrics_df = compute_metrics_df.round({
    "Params (M)": 3,
    "Model Size (MB)": 3,
    "Inference Latency (ms/img)": 3,
    "FLOPs/Image (G)": 3,
    "FLOPs/Batch (G)": 3,
    "Inference Memory Delta (MB)": 2,
})


# ============================================================
# DESCRIPTIVE STATS DATAFRAME
# ============================================================

descriptive_stats_df = pd.DataFrame({
    "Model": [MODEL_NAME],

    "Train Acc Mean": [compute_metrics["Train Stats"]["Accuracy Mean"]],
    "Train Acc Std": [compute_metrics["Train Stats"]["Accuracy Std"]],
    "Train Acc Median": [compute_metrics["Train Stats"]["Accuracy Median"]],
    "Train Acc IQR": [compute_metrics["Train Stats"]["Accuracy IQR"]],
    "Train Acc Range": [compute_metrics["Train Stats"]["Accuracy Range"]],

    "Train Loss Mean": [compute_metrics["Train Stats"]["Loss Mean"]],
    "Train Loss Std": [compute_metrics["Train Stats"]["Loss Std"]],
    "Train Loss Median": [compute_metrics["Train Stats"]["Loss Median"]],
    "Train Loss IQR": [compute_metrics["Train Stats"]["Loss IQR"]],
    "Train Loss Range": [compute_metrics["Train Stats"]["Loss Range"]],

    "Val Acc Mean": [compute_metrics["Validation Stats"]["Validation Accuracy Mean"]],
    "Val Acc Std": [compute_metrics["Validation Stats"]["Validation Accuracy Std"]],
    "Val Acc Median": [compute_metrics["Validation Stats"]["Validation Accuracy Median"]],
    "Val Acc IQR": [compute_metrics["Validation Stats"]["Validation Accuracy IQR"]],
    "Val Acc Range": [compute_metrics["Validation Stats"]["Validation Accuracy Range"]],

    "Val Loss Mean": [compute_metrics["Validation Stats"]["Validation Loss Mean"]],
    "Val Loss Std": [compute_metrics["Validation Stats"]["Validation Loss Std"]],
    "Val Loss Median": [compute_metrics["Validation Stats"]["Validation Loss Median"]],
    "Val Loss IQR": [compute_metrics["Validation Stats"]["Validation Loss IQR"]],
    "Val Loss Range": [compute_metrics["Validation Stats"]["Validation Loss Range"]],

    "Train Acc 95% CI": [compute_metrics["Train Stats"]["Accuracy CI"]],
    "Train Loss 95% CI": [compute_metrics["Train Stats"]["Loss CI"]],

    "Train Acc T-test p-value": [compute_metrics["Train Stats"]["Accuracy T-test"][1]],
    "Train Loss T-test p-value": [compute_metrics["Train Stats"]["Loss T-test"][1]],

    "Train Acc Cohen's d": [compute_metrics["Train Stats"]["Accuracy Effect Size"][0]],
    "Train Loss Cohen's d": [compute_metrics["Train Stats"]["Loss Effect Size"][0]],

    "Train Acc Normality p-value": [compute_metrics["Train Stats"]["Accuracy Normality Test"][1]],
    "Train Loss Normality p-value": [compute_metrics["Train Stats"]["Loss Normality Test"][1]],

    "Train Acc Wilcoxon p-value": [compute_metrics["Train Stats"]["Accuracy Wilcoxon Test"][1]],
    "Train Loss Wilcoxon p-value": [compute_metrics["Train Stats"]["Loss Wilcoxon Test"][1]],

    "Acc-Loss Pearson r": [compute_metrics["Train Stats"]["Accuracy-Loss Correlation"][0]],
    "Acc-Loss Pearson p-value": [compute_metrics["Train Stats"]["Accuracy-Loss Correlation"][1]],
    "Acc-Loss Spearman rho": [compute_metrics["Train Stats"]["Accuracy-Loss Correlation"][2]],
    "Acc-Loss Spearman p-value": [compute_metrics["Train Stats"]["Accuracy-Loss Correlation"][3]],
})

descriptive_stats_df = descriptive_stats_df.round(4)


# ============================================================
# FINAL COMBINED DATAFRAME
# ============================================================

final_df = pd.concat(
    [
        compute_metrics_df,
        descriptive_stats_df.drop(columns=["Model"])
    ],
    axis=1
)

display(final_df)


# ============================================================
# OPTIONAL: SAVE RESULTS
# ============================================================

final_df.to_csv("computational_cost_and_descriptive_statistics.csv", index=False)

print("\nSaved as: computational_cost_and_descriptive_statistics.csv")